# [August 2026] Kyber / FrodoKEM benchmarks and Q-Safe simulation

Final pipeline for the manuscript's Figure 3 (Kyber vs FrodoKEM), Figures 4-5 (Q-Safe scenarios), Table 2, and the Figure 6 schematic source.

**Run order matters** (Runtime → Run all): environment → install liboqs → write modules → benchmark suite → verification → Figure 3 → Q-Safe simulation.

The old 2025 exploratory cells were removed in this cleanup — they are preserved in the previous notebook version / git history. The RSA-vs-Kyber two-panel module (`plot_rsa_vs_kyber.py`) was removed because that figure was **dropped from the final figure set** (its data duplicates Figures 2 and 3).

# Section 1: Setup & Environment

In [1]:
# ============================================================
# 1. Record the execution environment (goes in Methods)
# ============================================================
!lscpu | head -8
!cat /etc/os-release | head -2
import sys, platform
print("Python:", sys.version)

Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           48 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Vendor ID:                               AuthenticAMD
Model name:                              AMD EPYC 7B12
PRETTY_NAME="Ubuntu 24.04.5 LTS"
NAME="Ubuntu"
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
# ============================================================
# Optimized Parallel Build & Google Drive Fast-Cache Setup
# ============================================================
import os
import platform
import shutil
import subprocess
import sys
import time
import json
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

def run_shell(cmd: str, timeout: int | None = None) -> None:
    """
    Run a shell command with live-streamed output (unlike os.system, which
    can buffer output until the process exits) and raise with diagnostics
    if it fails or times out.
    """
    print(f"$ {cmd}")
    process = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    tail = []
    timed_out = False
    start = time.time()
    try:
        for line in process.stdout:
            print(line, end="")
            sys.stdout.flush()
            tail.append(line)
            tail = tail[-50:]
            if timeout is not None and (time.time() - start) > timeout:
                process.kill()
                timed_out = True
                break
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        timed_out = True

    if timed_out:
        print(f"\n--- Diagnostics (timeout after {timeout}s) ---")
        os.system("free -h")
        os.system("df -h /root /content")
        raise RuntimeError(f"Command timed out after {timeout}s: {cmd}")

    if process.returncode != 0:
        print("\n--- Diagnostics ---")
        os.system("free -h")
        os.system("df -h /root /content")
        print("--- Last output lines ---")
        print("".join(tail))
        raise RuntimeError(f"Command failed (exit {process.returncode}): {cmd}")

def run_cpu_sanity_check(threshold_seconds: float = 0.5, retries: int = 3) -> bool:
    """
    Rough check that the host isn't badly CPU-throttled.
    Uses a numpy matmul (real compute) instead of a Python loop,
    and retries a couple times before failing, since shared vCPUs
    can have transient blips that aren't true throttling.
    """
    import numpy as np
    print("--- Running Host CPU Sanity Check ---")
    for attempt in range(1, retries + 1):
        a = np.random.rand(600, 600)
        b = np.random.rand(600, 600)
        t0 = time.perf_counter()
        a @ b
        elapsed = time.perf_counter() - t0

        if elapsed <= threshold_seconds:
            print(f"[PASS] Sanity compute check: {elapsed:.3f}s < {threshold_seconds:.2f}s (attempt {attempt})")
            return True

        print(f"[WARN] Attempt {attempt}/{retries}: {elapsed:.3f}s (Threshold: < {threshold_seconds:.2f}s)")
        time.sleep(1)

    print("-> Host VM appears CPU-throttled after retries. Disconnect and recreate the Colab runtime.")
    return False

# 2. Host CPU Sanity Check
assert run_cpu_sanity_check(), "Host CPU check failed. Recycle runtime before building."

# 3. Setup constants
drive_cache = "/content/drive/MyDrive/liboqs_cache/_oqs"
target_dir = "/root/_oqs"
manifest_name = "_build_manifest.json"
# Cap parallel compile jobs at half the visible cores to reduce OOM risk
# (liboqs spawns many memory-hungry GCC jobs; unmetered -j has stalled
# for 10-15 min before failing on some Colab VMs).
BUILD_JOBS = max(1, (os.cpu_count() or 2) // 2)

def current_manifest() -> dict:
    """Fingerprint of the environment the cache was built on."""
    return {
        "platform": platform.platform(),
        "machine": platform.machine(),
        "python_version": platform.python_version(),
        "libc": " ".join(platform.libc_ver()),
    }

def cache_is_compatible(cache_dir: str) -> bool:
    manifest_path = os.path.join(cache_dir, manifest_name)
    if not os.path.exists(manifest_path):
        print("[CACHE] No manifest found alongside cache; treating as incompatible.")
        return False
    try:
        with open(manifest_path) as f:
            cached = json.load(f)
    except Exception as e:
        print(f"[CACHE] Failed to read manifest ({e}); treating as incompatible.")
        return False
    current = current_manifest()
    if cached != current:
        print("[CACHE] Environment fingerprint mismatch; treating cache as stale.")
        print(f"  cached : {cached}")
        print(f"  current: {current}")
        return False
    return True

def build_liboqs_from_source() -> None:
    print(f"\n[BUILD] Compiling liboqs C library using Ninja (-j{BUILD_JOBS})...")
    run_shell("apt-get update -qq")
    run_shell("apt-get install -y -qq cmake ninja-build build-essential")

    # Start from a clean checkout to avoid stale CMakeCache.txt from a
    # previous failed/partial build.
    if os.path.exists("liboqs"):
        shutil.rmtree("liboqs")
    run_shell("git clone --depth=1 https://github.com/open-quantum-safe/liboqs.git")

    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)

    cwd = os.getcwd()
    os.chdir("liboqs")
    try:
        run_shell(f"cmake -GNinja -DCMAKE_INSTALL_PREFIX={target_dir} -DBUILD_SHARED_LIBS=ON .")
        run_shell(f"ninja -j{BUILD_JOBS} install", timeout=900)
    finally:
        os.chdir(cwd)

    if not os.path.isdir(target_dir):
        raise RuntimeError("Build reported success but install dir is missing.")

    # Write a manifest so we know if this build is safe to reuse later.
    with open(os.path.join(target_dir, manifest_name), "w") as f:
        json.dump(current_manifest(), f)

    print("Caching compiled binaries to Google Drive for future resets...")
    os.makedirs(os.path.dirname(drive_cache), exist_ok=True)
    if os.path.exists(drive_cache):
        shutil.rmtree(drive_cache)
    # symlinks=True preserves the versioned .so symlinks liboqs installs,
    # instead of tripling storage with independent full copies.
    shutil.copytree(target_dir, drive_cache, symlinks=True)

def restore_from_cache() -> bool:
    print("\n[FAST RESTORE] Attempting to restore compiled liboqs from Google Drive (~5s)...")
    os.makedirs("/root", exist_ok=True)
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)
    shutil.copytree(drive_cache, target_dir, symlinks=True)
    return cache_is_compatible(target_dir)

# 4. Check Google Drive for cached compiled C library
restored_ok = False
if os.path.exists(drive_cache):
    restored_ok = restore_from_cache()
    if not restored_ok:
        print("[CACHE] Restored cache failed compatibility check; deleting and rebuilding.")
        if os.path.exists(target_dir):
            shutil.rmtree(target_dir)
        shutil.rmtree(drive_cache, ignore_errors=True)

if not restored_ok:
    build_liboqs_from_source()

# Make the install location explicit rather than relying on liboqs-python's
# default ~/_oqs convention (works today because Colab runs as root, but
# don't rely on that implicitly).
os.environ["LIBOQS_INSTALL_PATH"] = target_dir

# 5. Install liboqs Python bindings
if os.path.exists("liboqs-python"):
    shutil.rmtree("liboqs-python")
run_shell("git clone --depth=1 https://github.com/open-quantum-safe/liboqs-python")

cwd = os.getcwd()
os.chdir("liboqs-python")
try:
    # No --no-build-isolation: let pip read pyproject.toml and install
    # the declared build backend (hatchling) into an isolated build env
    # automatically, rather than assuming it's already present.
    run_shell("pip install .")
finally:
    os.chdir(cwd)

# 6. Native verification
try:
    import oqs
except OSError as e:
    print(f"[FAIL] Failed to load liboqs bindings: {e}")
    print("-> Cached binaries likely incompatible with this host. Clearing cache; rerun the cell to rebuild.")
    shutil.rmtree(drive_cache, ignore_errors=True)
    raise

print(f"\n[SUCCESS] Environment Ready on Clean Host!")
print(f"liboqs version: {oqs.oqs_version()}")
print(f"liboqs-python version: {oqs.oqs_python_version()}")

MessageError: Error: credential propagation was unsuccessful

# Section 2: Configuration & Validation Modules

In [1]:
%%writefile pqc_config.py
"""
pqc_config.py

Single source of truth for the Kyber / FrodoKEM benchmark run parameters --
the KEM counterpart of run_config.py on the RSA side.

WHY THIS FILE EXISTS
--------------------
Before it, the trial counts lived in two unconnected places:

  * the notebook called run_full_benchmark_suite(num_trials=100,
    kyber_num_trials=500), and
  * verify_pqc_results.py separately hardcoded KYBER_TRIALS = 500 and
    FRODO_TRIALS = 100.

Nothing tied those together. Change one and forget the other and the
verification either fails on data that was actually fine, or -- worse --
passes while checking the wrong thing. The benchmark, the pre-flight
check and the post-run assertions now all read the constants below, so
the numbers being asserted can never drift from the numbers that ran.

This is exactly the role run_config.py plays for RSA. Same reason, same
shape.

CHANGING THESE VALUES INVALIDATES THE PUBLISHED DATA. The manuscript's
Table 2, Figures 3-5 and every quoted per-exchange time come from one
frozen run at the values below. Edit them only if you intend to re-run
the whole KEM pipeline and re-derive every downstream number.
"""

# --- Algorithm sets -----------------------------------------------------
# liboqs mechanism names, exactly as oqs.get_supported_kem_mechanisms()
# reports them. A typo here surfaces in the pre-flight check rather than
# an hour into the run.
KYBER_ALGS = ["Kyber512", "Kyber768", "Kyber1024"]
FRODO_ALGS = ["FrodoKEM-640-AES", "FrodoKEM-976-AES", "FrodoKEM-1344-AES"]

# The matched-security-level pair used for the complete-key-exchange
# comparison and as the Q-Safe simulation's timing inputs. Both are NIST
# security level 3 -- that matching is the entire basis for the "54.6x"
# claim, so it is named here rather than buried in a function body.
LEVEL3_KYBER = "Kyber768"
LEVEL3_FRODO = "FrodoKEM-976-AES"

# --- Trial counts -------------------------------------------------------
# Kyber operations run in the tens of microseconds, where OS scheduling
# jitter is a large fraction of the signal; FrodoKEM's are milliseconds,
# where it is not. More Kyber trials tighten its interval at almost no
# wall-clock cost, so the two families are sampled differently on purpose.
# (Same reasoning as RSA-1024's 500 samples in run_config.py.)
KYBER_TRIALS = 500
FRODO_TRIALS = 100

# Untimed calls discarded before measurement begins. The first operation
# against a fresh key costs noticeably more than a steady-state one; left
# in, that artifact attaches itself to whichever measurement happened to
# run first.
NUM_WARMUP = 5

# Plaintext length handed to the KEM harness, in bytes. KEM ciphertext
# sizes do not depend on it -- it is fixed only so the runs are identical.
MESSAGE_LENGTH_BYTES = 64

# --- Output paths -------------------------------------------------------
# CSVs stay at the repository root (matching rsa_benchmark_results.csv);
# every generated image goes to FIGURE_DIR, which the scripts create.
PER_VARIANT_CSV = "pqc_benchmark_results.csv"
COMPLETE_EXCHANGE_CSV = "pqc_complete_key_exchange_results.csv"
TABLE2_CSV = "Table 2 - Q-Safe Simulation Results.csv"
FIGURE_DIR = "figures"


def expected_row_counts() -> dict:
    """
    Row counts each CSV must contain if the run completed correctly.
    Returned as data rather than printed, so the pre-flight check, the
    post-run assertions and any ad-hoc inspection all quote the same
    numbers.
    """
    per_variant = {alg: KYBER_TRIALS for alg in KYBER_ALGS}
    per_variant.update({alg: FRODO_TRIALS for alg in FRODO_ALGS})
    complete = {LEVEL3_KYBER: KYBER_TRIALS, LEVEL3_FRODO: FRODO_TRIALS}
    return {"per_variant": per_variant, "complete_key_exchange": complete}

Writing pqc_config.py


In [2]:
%%writefile verify_pqc_results.py
"""
verify_pqc_results.py

Pre-flight configuration checks and post-run data assertions for the
Kyber / FrodoKEM benchmarks -- the KEM counterpart of
verify_rsa_results.py.

Two entry points, mirroring the RSA side:

    preflight_check()      run BEFORE the benchmark. Validates the config
                           and (if liboqs is installed) that every
                           mechanism named in pqc_config.py is actually
                           supported by this build. Catches a typo or a
                           liboqs built without FrodoKEM in two seconds
                           instead of an hour into the run.

    verify_pqc_results()   run AFTER the benchmark, on the two CSVs.
                           Trial counts, missing values, ciphertext
                           constants, FrodoKEM variant detection.

Standalone use:

    python verify_pqc_results.py          # pre-flight + verify the committed CSVs

All expected values come from pqc_config.py. Nothing in this file
restates a run parameter -- that duplication is exactly what caused the
drift this file now guards against.
"""
import os

import pandas as pd

from pqc_config import (
    KYBER_ALGS, FRODO_ALGS, LEVEL3_KYBER, LEVEL3_FRODO,
    KYBER_TRIALS, FRODO_TRIALS, NUM_WARMUP, MESSAGE_LENGTH_BYTES,
    PER_VARIANT_CSV, COMPLETE_EXCHANGE_CSV, expected_row_counts,
)

# Ciphertext sizes are protocol constants -- the canary for column drift
# and for a silently changed algorithm variant. They are verification
# constants, not run parameters, which is why they live here and not in
# pqc_config.py. Kyber's are stable across liboqs versions; FrodoKEM's
# depend on which variant liboqs ships: pre-ISO (unsalted) vs.
# ISO-standardized (salted, +32/+48/+64 bytes).
KYBER_CT = {"Kyber512": 768, "Kyber768": 1088, "Kyber1024": 1568}
FRODO_CT_PRE_ISO = {"FrodoKEM-640-AES": 9720, "FrodoKEM-976-AES": 15744,
                    "FrodoKEM-1344-AES": 21632}
FRODO_CT_ISO = {"FrodoKEM-640-AES": 9752, "FrodoKEM-976-AES": 15792,
                "FrodoKEM-1344-AES": 21696}


def preflight_check(verbose: bool = True) -> dict:
    """
    Validate the run configuration BEFORE spending an hour measuring.
    Returns the expected row counts so the caller does not have to
    recompute them.

    Checks, in order of how much time each one saves:

      1. liboqs actually supports every mechanism named in the config.
         A liboqs built without FrodoKEM enabled, or a mechanism name
         that changed between versions, fails here in seconds rather
         than after the Kyber half has already run.
      2. Trial counts are positive integers.
      3. The level-3 pair named for the complete-key-exchange comparison
         is present in the per-variant algorithm lists -- otherwise the
         two CSVs describe different algorithm sets.
      4. Warm-up count is non-negative and smaller than the trial count.

    Skips check 1 when liboqs is not installed, so this function still
    runs in a --reuse-csv workflow on a machine that never built liboqs.
    """
    counts = expected_row_counts()

    assert isinstance(KYBER_TRIALS, int) and KYBER_TRIALS > 0, \
        f"KYBER_TRIALS must be a positive int, got {KYBER_TRIALS!r}"
    assert isinstance(FRODO_TRIALS, int) and FRODO_TRIALS > 0, \
        f"FRODO_TRIALS must be a positive int, got {FRODO_TRIALS!r}"
    assert 0 <= NUM_WARMUP < min(KYBER_TRIALS, FRODO_TRIALS), \
        f"NUM_WARMUP ({NUM_WARMUP}) must be >= 0 and smaller than the trial counts"
    assert MESSAGE_LENGTH_BYTES > 0, "MESSAGE_LENGTH_BYTES must be positive"

    assert LEVEL3_KYBER in KYBER_ALGS, \
        f"LEVEL3_KYBER ({LEVEL3_KYBER}) is not in KYBER_ALGS -- the two CSVs " \
        f"would cover different algorithm sets"
    assert LEVEL3_FRODO in FRODO_ALGS, \
        f"LEVEL3_FRODO ({LEVEL3_FRODO}) is not in FRODO_ALGS -- the two CSVs " \
        f"would cover different algorithm sets"

    # Mechanism-support check. Import is local and guarded so this module
    # stays usable without liboqs.
    try:
        import oqs
    except ImportError:
        oqs = None

    if oqs is None:
        if verbose:
            print("liboqs not installed -- skipping the mechanism-support check.")
            print("(Fine for --reuse-csv; a live benchmark run needs liboqs.)")
    else:
        supported = set(oqs.get_supported_kem_mechanisms())
        missing = [a for a in KYBER_ALGS + FRODO_ALGS if a not in supported]
        assert not missing, (
            f"liboqs does not support: {missing}. Either the mechanism names in "
            f"pqc_config.py are wrong for this liboqs version, or this build was "
            f"compiled without them. Check oqs.get_supported_kem_mechanisms()."
        )
        if verbose:
            print(f"liboqs {oqs.oqs_version()} supports all "
                  f"{len(KYBER_ALGS) + len(FRODO_ALGS)} required mechanisms.")

    if verbose:
        total_pv = sum(counts["per_variant"].values())
        total_cx = sum(counts["complete_key_exchange"].values())
        print(f"\nPre-flight configuration check")
        print(f"  Kyber variants:   {', '.join(KYBER_ALGS)} @ {KYBER_TRIALS} trials")
        print(f"  FrodoKEM variants: {', '.join(FRODO_ALGS)} @ {FRODO_TRIALS} trials")
        print(f"  Level-3 pair:      {LEVEL3_KYBER} vs {LEVEL3_FRODO}")
        print(f"  Warm-ups discarded: {NUM_WARMUP} per algorithm")
        print(f"  Expected rows: {total_pv} in {PER_VARIANT_CSV}, "
              f"{total_cx} in {COMPLETE_EXCHANGE_CSV}")
        print("  Pre-flight checks passed.\n")

    return counts


def verify_pqc_results(per_variant_csv: str = PER_VARIANT_CSV,
                       complete_csv: str = COMPLETE_EXCHANGE_CSV):
    """Post-run assertions on the two CSVs. Raises on any inconsistency."""
    for path in (per_variant_csv, complete_csv):
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"{path} not found. Run the benchmark first "
                f"(python run_pqc_benchmark.py), or point this function at the "
                f"committed CSV."
            )

    expected = expected_row_counts()
    pv = pd.read_csv(per_variant_csv)

    # 1. Trial counts per variant -- expectations come from pqc_config.py,
    #    so they cannot disagree with what the benchmark was told to do.
    counts = pv.groupby("algorithm").size()
    for alg, n_expected in expected["per_variant"].items():
        assert counts.get(alg, 0) == n_expected, \
            f"{alg}: {counts.get(alg, 0)} trials in CSV, expected {n_expected}"

    # 2. No missing timings
    assert pv[["keygen_time_s", "encap_time_s", "decap_time_s"]].isna().sum().sum() == 0, \
        "NaN timings present -- a trial failed silently"

    # 3. Ciphertext constants, and detect which FrodoKEM variant liboqs shipped
    ct = pv.groupby("algorithm")["ciphertext_size_bytes"].agg(lambda s: set(s))
    for alg, expected_ct in KYBER_CT.items():
        assert ct[alg] == {expected_ct}, \
            f"{alg} ciphertext {ct[alg]}, expected {{{expected_ct}}}"

    frodo_measured = {alg: ct[alg] for alg in FRODO_ALGS}
    if all(frodo_measured[a] == {FRODO_CT_ISO[a]} for a in FRODO_ALGS):
        variant = "ISO-standardized FrodoKEM (salted ciphertexts)"
    elif all(frodo_measured[a] == {FRODO_CT_PRE_ISO[a]} for a in FRODO_ALGS):
        variant = "pre-ISO FrodoKEM (unsalted ciphertexts)"
    else:
        raise AssertionError(
            f"FrodoKEM ciphertexts match neither known variant: {frodo_measured}")
    print(f"FrodoKEM variant detected: {variant}")
    print("STATE THIS IN METHODS along with the liboqs version -- the ciphertext sizes")
    print("in the manuscript must match this variant, not the other one.")

    # 4. Complete-key-exchange medians -- the Q-Safe inputs and Table 2 basis
    cx = pd.read_csv(complete_csv)
    cx_counts = cx.groupby("algorithm").size()
    for alg, n_expected in expected["complete_key_exchange"].items():
        assert cx_counts.get(alg, 0) == n_expected, \
            f"{alg} (complete exchange): {cx_counts.get(alg, 0)} trials, expected {n_expected}"

    med = cx.groupby("algorithm")["total_time_s"].median()
    k, f = med[LEVEL3_KYBER] * 1000, med[LEVEL3_FRODO] * 1000
    print(f"\nComplete key exchange (median): {LEVEL3_KYBER} {k:.4f} ms | "
          f"{LEVEL3_FRODO} {f:.4f} ms | ratio {f / k:.1f}x")
    print("These are the ONLY authoritative per-exchange values -- Table 2, the Q-Safe")
    print("figures, and every text claim must use these, not docstring examples.")
    print("\nAll PQC verification assertions passed.")

    return pv, cx


if __name__ == "__main__":
    preflight_check()
    verify_pqc_results()


Writing verify_pqc_results.py


# Section 3: Benchmarking & Core Engine Modules

In [3]:
%%writefile benchmark_pqc.py
"""
benchmark_pqc.py

Kyber and FrodoKEM benchmarking via liboqs-python.

SETUP (run this in a Colab cell BEFORE running this script -- `pip install
oqs` on its own does NOT work, because there is no prebuilt wheel; the
`oqs` Python bindings require building the liboqs C library from source):

    !git clone --depth=1 https://github.com/open-quantum-safe/liboqs-python
    %cd liboqs-python
    !pip install .
    %cd ..

This clones the liboqs-python repo (which bundles a copy of the liboqs C
library), builds it via CMake, and installs the Python bindings. It takes
a few minutes the first time. Colab already has the build tools (cmake,
ninja, a C compiler) preinstalled; if you're running this somewhere else
and the `pip install .` step fails with a missing-cmake/ninja error, run:

    !apt-get install -y cmake ninja-build

first, then retry the liboqs-python install.

Run this in Google Colab (or any environment with liboqs-python installed
per the above). It requires network access to install liboqs, so it will
NOT run in this sandbox -- copy it into Colab, run the setup cell above
first, then run this script in a separate cell (or `python benchmark_pqc.py`
in a `!` shell cell) in the same session.

Fixes applied relative to the original RSA_Kyber_Benchmarking.ipynb:

1. ORIGINAL BUG: `benchmark_pqc()` took exactly ONE time.time() sample per
   (algorithm, message_length) pair. The paper's Methods section claims
   "100 trials were conducted, and the average values were used" -- that
   was never actually true for this function. Fixed: now averages
   `num_trials` (default 100) runs per algorithm, using time.perf_counter()
   for better sub-millisecond resolution, and reports both mean and median
   (median is more robust to occasional OS scheduling spikes).

2. ORIGINAL BUG: the `message_lengths = [1024, 2048, 4096]` loop is
   meaningless for KEM operations. `encap_secret()`/`decap_secret()` don't
   take a message as input at all -- KEM output size and timing depend
   only on the algorithm/security level, not on a "message length." The
   original code silently looped over the same operation 3x, adding noise
   without adding information. Fixed: removed. If you want three data
   points per algorithm for a chart, that's what the three separate
   Kyber/FrodoKEM security-level variants already give you.

3. ORIGINAL GAP: no code ever printed/saved the actual liboqs version used,
   even though your Methods text has a literal placeholder
   "[liboqs version and KEM identifier as printed by the benchmark
   notebook]". Fixed: version and mechanism list are captured into the
   results dict and printed, ready to paste into Methods.

4. NEW: added `benchmark_complete_key_exchange()`, which measures
   keygen + encapsulation + decapsulation as a single combined operation,
   median of 100 trials, for Kyber768 and FrodoKEM-976-AES specifically
   (NIST security level 3). This is the number your Q-Safe cost model and
   the per-exchange comparison reported in Table 2 of the manuscript --
   it did not previously exist as a standalone, clearly-scoped function.
"""

import time
import statistics
import platform

import pandas as pd

# liboqs is only needed to MEASURE. load_results_from_csv() rebuilds the
# same result structure from the committed CSVs, and every plotting and
# simulation script downstream works from that -- so importing this
# module must not hard-fail on a machine that never built liboqs.
# Without this guard, `run_pqc_benchmark.py --reuse-csv` (whose entire
# purpose is to regenerate figures without re-measuring) could not run
# outside Colab.
try:
    import oqs
except ImportError:
    oqs = None

from pqc_config import (
    KYBER_ALGS, FRODO_ALGS, LEVEL3_KYBER, LEVEL3_FRODO,
    KYBER_TRIALS, FRODO_TRIALS, NUM_WARMUP, MESSAGE_LENGTH_BYTES,
    PER_VARIANT_CSV, COMPLETE_EXCHANGE_CSV,
)


def _require_oqs():
    """Fail with an actionable message, not a bare NameError, when a
    measuring function is called without liboqs installed."""
    if oqs is None:
        raise ImportError(
            "liboqs-python is not installed, so nothing can be measured.\n"
            "  Install:  git clone --depth=1 https://github.com/open-quantum-safe/liboqs-python\n"
            "            cd liboqs-python && pip install . && cd ..\n"
            "  Or, to work from the committed data instead of measuring:\n"
            "            python run_pqc_benchmark.py --reuse-csv"
        )


def get_environment_info():
    """Capture the exact liboqs version and platform for the Methods section."""
    _require_oqs()
    info = {
        "liboqs_version": oqs.oqs_version(),
        "liboqs_python_version": oqs.oqs_python_version(),
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "supported_kem_mechanisms": oqs.get_supported_kem_mechanisms(),
    }
    print("liboqs version:", info["liboqs_version"])
    print("liboqs-python version:", info["liboqs_python_version"])
    print("Python version:", info["python_version"])
    print("Platform:", info["platform"])
    return info


def benchmark_kem(kem_alg: str, num_trials: int = FRODO_TRIALS,
                   message_length: int = MESSAGE_LENGTH_BYTES,
                   num_warmup: int = NUM_WARMUP):
    """
    Benchmark key generation, encapsulation, and decapsulation for a single
    KEM algorithm, averaged over num_trials independent runs.

    `num_warmup` untimed trials are run first and discarded. This avoids
    contaminating the timed samples with one-time costs (first-call
    interpreter/library warm-up, cold caches) that don't reflect steady-
    state performance -- this matters more for fast operations (Kyber, at
    tens of microseconds) than slow ones, since a fixed warm-up cost is a
    much larger fraction of a small measurement.

    `message_length` is accepted only for logging/consistency with the RSA
    benchmark script's parameterization -- it has NO effect on timing or
    output size for a KEM, since encap_secret()/decap_secret() do not take
    a plaintext message. It is recorded in the result dict for traceability
    but should not be looped over as an independent variable.

    Returns a dict with:
      - mean/median/min/stdev/sem summaries per metric (as before, for
        direct compatibility with plot_pqc_comparison.py)
      - "raw" sub-dict with the individual per-trial timing lists
        (keygen_time_s, encap_time_s, decap_time_s), so results can be
        saved to a tidy CSV and re-aggregated later with any stat,
        exactly like benchmark_rsa.py's per-sample rows. This also means
        the raw data can be independently re-checked later (e.g. for the
        same kind of contention/bimodal timing issues found in the RSA
        benchmarks) without re-running the benchmark.

    NOTE ON INTERPRETING RESULTS: for very fast operations (Kyber's ~tens
    of microseconds), OS scheduling jitter and Python-level overhead can
    dominate the measurement, especially on a shared/virtualized
    environment like Colab. If the mean/median for adjacent security
    levels (e.g. Kyber768 vs Kyber1024) look nearly identical with large
    overlapping error bars, that is often a real noise-floor effect, not
    a bug. The `min` across trials is a diagnostic worth glancing at in
    that situation (system interference only ever adds delay, never
    subtracts it -- the principle behind Python's `timeit` reporting the
    minimum), but THIS PROJECT'S REPORTING CONVENTION IS MEDIAN
    throughout -- for the manuscript, all figures/tables, and the Q-Safe
    simulation inputs -- chosen for robustness to the transient
    contention measured on Colab and for consistency with the RSA
    benchmarks. Don't switch any single artifact to min/mean without
    switching all of them and the Methods text together.
    """
    keygen_times = []
    encap_times = []
    decap_times = []
    ciphertext_size = None

    kem = oqs.KeyEncapsulation(kem_alg)

    for _ in range(num_warmup):
        public_key = kem.generate_keypair()
        ciphertext, _ = kem.encap_secret(public_key)
        kem.decap_secret(ciphertext)

    for _ in range(num_trials):
        start = time.perf_counter()
        public_key = kem.generate_keypair()
        keygen_times.append(time.perf_counter() - start)

        start = time.perf_counter()
        ciphertext, shared_secret = kem.encap_secret(public_key)
        encap_times.append(time.perf_counter() - start)
        ciphertext_size = len(ciphertext)

        start = time.perf_counter()
        recovered_secret = kem.decap_secret(ciphertext)
        decap_times.append(time.perf_counter() - start)

        assert shared_secret == recovered_secret, f"Shared secret mismatch for {kem_alg}!"

    def summarize(samples):
        n = len(samples)
        stdev = statistics.stdev(samples) if n > 1 else 0.0
        return {
            "mean": statistics.mean(samples),
            "median": statistics.median(samples),
            "min": min(samples),
            "stdev": stdev,
            "sem": stdev / (n ** 0.5) if n > 1 else 0.0,
        }

    return {
        "algorithm": kem_alg,
        "message_length_note": message_length,  # recorded, not varied
        "num_trials": num_trials,
        "keygen_time_s": summarize(keygen_times),
        "encap_time_s": summarize(encap_times),
        "decap_time_s": summarize(decap_times),
        "ciphertext_size_bytes": ciphertext_size,
        "raw": {
            "keygen_time_s": keygen_times,
            "encap_time_s": encap_times,
            "decap_time_s": decap_times,
        },
    }


def benchmark_complete_key_exchange(kem_alg: str, num_trials: int = FRODO_TRIALS,
                                    num_warmup: int = NUM_WARMUP):
    """
    Measure a COMPLETE key exchange (keygen + encapsulation + decapsulation)
    as a single combined timed operation, per trial. This is the quantity
    the Q-Safe cost model in the paper actually uses (the per-exchange medians reported in Table 2 of the manuscript).

    Returns mean/median/min/stdev/sem in seconds, plus a "raw" list of the
    individual per-trial total times -- see benchmark_kem()'s docstring
    for why raw per-trial data is saved rather than only summaries.
    """
    kem = oqs.KeyEncapsulation(kem_alg)

    for _ in range(num_warmup):
        public_key = kem.generate_keypair()
        ciphertext, _ = kem.encap_secret(public_key)
        kem.decap_secret(ciphertext)

    total_times = []
    for _ in range(num_trials):
        start = time.perf_counter()
        public_key = kem.generate_keypair()
        ciphertext, shared_secret = kem.encap_secret(public_key)
        recovered_secret = kem.decap_secret(ciphertext)
        total_times.append(time.perf_counter() - start)
        assert shared_secret == recovered_secret

    n = len(total_times)
    stdev = statistics.stdev(total_times) if n > 1 else 0.0
    return {
        "algorithm": kem_alg,
        "num_trials": num_trials,
        "mean_time_s": statistics.mean(total_times),
        "median_time_s": statistics.median(total_times),
        "min_time_s": min(total_times),
        "stdev_time_s": stdev,
        "sem_time_s": stdev / (n ** 0.5) if n > 1 else 0.0,
        "raw": {"total_time_s": total_times},
    }


def save_results_to_csv(results: dict, per_variant_csv: str = PER_VARIANT_CSV,
                          complete_exchange_csv: str = COMPLETE_EXCHANGE_CSV):
    """
    Flatten run_full_benchmark_suite()'s raw per-trial data into two tidy
    CSVs, matching the pattern already used for RSA (rsa_benchmark_results.csv):
    one row per trial, aggregate later with pandas groupby + whatever stat
    you want, rather than locking in mean/median/min at benchmark time.

    This decouples the (expensive, network-dependent, liboqs-only-runs-in-
    Colab) benchmarking step from the (fast, portable) analysis/plotting
    step -- once you have these CSVs, plotting and table generation can
    happen anywhere pandas/matplotlib are available, no liboqs required,
    and the raw data can be independently re-checked later (e.g. for the
    same kind of contention/bimodal timing issues found in the RSA
    benchmarks) without re-running the benchmark.

    per_variant_csv columns: algorithm, trial_idx, keygen_time_s,
    encap_time_s, decap_time_s, ciphertext_size_bytes

    complete_exchange_csv columns: algorithm, trial_idx, total_time_s
    """
    per_variant_rows = []
    for alg, data in results["per_variant"].items():
        raw = data["raw"]
        n = data["num_trials"]
        for i in range(n):
            per_variant_rows.append({
                "algorithm": alg,
                "trial_idx": i,
                "keygen_time_s": raw["keygen_time_s"][i],
                "encap_time_s": raw["encap_time_s"][i],
                "decap_time_s": raw["decap_time_s"][i],
                "ciphertext_size_bytes": data["ciphertext_size_bytes"],
            })
    per_variant_df = pd.DataFrame(per_variant_rows)
    per_variant_df.to_csv(per_variant_csv, index=False)
    print(f"Saved {len(per_variant_df)} rows to {per_variant_csv}")

    complete_rows = []
    for alg, data in results["complete_key_exchange"].items():
        raw = data["raw"]
        n = data["num_trials"]
        for i in range(n):
            complete_rows.append({
                "algorithm": alg,
                "trial_idx": i,
                "total_time_s": raw["total_time_s"][i],
            })
    complete_df = pd.DataFrame(complete_rows)
    complete_df.to_csv(complete_exchange_csv, index=False)
    print(f"Saved {len(complete_df)} rows to {complete_exchange_csv}")

    return per_variant_df, complete_df


def load_results_from_csv(per_variant_csv: str = PER_VARIANT_CSV,
                            complete_exchange_csv: str = COMPLETE_EXCHANGE_CSV):
    """
    Reconstruct the same nested dict structure run_full_benchmark_suite()
    returns (mean/median/min/stdev/sem per metric, plus "raw" per-trial
    lists) by reading back the tidy CSVs saved by save_results_to_csv().

    This means plot_pqc_comparison.py's plot_kem_comparison_panels()/plot_all_comparisons()
    work completely unchanged whether fed a live benchmark run or data
    loaded from disk -- e.g.:

        results = load_results_from_csv()
        plot_kem_comparison_panels(results["per_variant"], stat="median", error_bar_type="sem")

    No liboqs/Colab required for this half of the workflow.
    """
    def summarize(samples):
        n = len(samples)
        stdev = statistics.stdev(samples) if n > 1 else 0.0
        return {
            "mean": statistics.mean(samples),
            "median": statistics.median(samples),
            "min": min(samples),
            "stdev": stdev,
            "sem": stdev / (n ** 0.5) if n > 1 else 0.0,
        }

    per_variant_df = pd.read_csv(per_variant_csv)
    per_variant_results = {}
    for alg, group in per_variant_df.groupby("algorithm"):
        per_variant_results[alg] = {
            "algorithm": alg,
            "num_trials": len(group),
            "keygen_time_s": summarize(group["keygen_time_s"].tolist()),
            "encap_time_s": summarize(group["encap_time_s"].tolist()),
            "decap_time_s": summarize(group["decap_time_s"].tolist()),
            "ciphertext_size_bytes": group["ciphertext_size_bytes"].iloc[0],
            "raw": {
                "keygen_time_s": group["keygen_time_s"].tolist(),
                "encap_time_s": group["encap_time_s"].tolist(),
                "decap_time_s": group["decap_time_s"].tolist(),
            },
        }

    complete_df = pd.read_csv(complete_exchange_csv)
    complete_key_exchange_results = {}
    for alg, group in complete_df.groupby("algorithm"):
        times = group["total_time_s"].tolist()
        n = len(times)
        stdev = statistics.stdev(times) if n > 1 else 0.0
        complete_key_exchange_results[alg] = {
            "algorithm": alg,
            "num_trials": n,
            "mean_time_s": statistics.mean(times),
            "median_time_s": statistics.median(times),
            "min_time_s": min(times),
            "stdev_time_s": stdev,
            "sem_time_s": stdev / (n ** 0.5) if n > 1 else 0.0,
            "raw": {"total_time_s": times},
        }

    return {
        "per_variant": per_variant_results,
        "complete_key_exchange": complete_key_exchange_results,
    }


def run_full_benchmark_suite(num_trials: int = FRODO_TRIALS,
                               kyber_num_trials: int = KYBER_TRIALS,
                               save_csv: bool = True):
    """
    Runs the full Kyber vs. FrodoKEM benchmark suite matching the paper's
    reported metrics: per-variant keygen/encap/decap/ciphertext-size, and
    the NIST-level-3 complete-key-exchange comparison used by Q-Safe.

    `kyber_num_trials`: if set, overrides num_trials for Kyber specifically.
    Kyber operations run in the tens-of-microseconds range, where OS
    scheduling jitter is a larger fraction of the signal than for
    FrodoKEM's millisecond-scale operations -- more trials (e.g. 500-1000)
    tighten Kyber's confidence interval without materially increasing
    total runtime, since each trial is still very fast. Defaults to
    num_trials if not set.
    """
    _require_oqs()
    env_info = get_environment_info()

    # Algorithm sets and trial counts come from pqc_config.py -- the same
    # module verify_pqc_results.py checks against, so the assertions can
    # never drift from what actually ran.
    kyber_algs = KYBER_ALGS
    frodo_algs = FRODO_ALGS
    kyber_trials = kyber_num_trials if kyber_num_trials is not None else num_trials

    per_variant_results = {}
    for alg in kyber_algs:
        print(f"Benchmarking {alg} ({kyber_trials} trials)...")
        per_variant_results[alg] = benchmark_kem(alg, num_trials=kyber_trials)
    for alg in frodo_algs:
        print(f"Benchmarking {alg} ({num_trials} trials)...")
        per_variant_results[alg] = benchmark_kem(alg, num_trials=num_trials)

    print("\nBenchmarking complete key exchange at NIST security level 3...")
    complete_exchange_results = {
        LEVEL3_KYBER: benchmark_complete_key_exchange(LEVEL3_KYBER, num_trials=kyber_trials),
        LEVEL3_FRODO: benchmark_complete_key_exchange(LEVEL3_FRODO, num_trials=num_trials),
    }

    kyber_min_ms = complete_exchange_results[LEVEL3_KYBER]["min_time_s"] * 1000
    frodo_min_ms = complete_exchange_results[LEVEL3_FRODO]["min_time_s"] * 1000
    kyber_med_ms = complete_exchange_results[LEVEL3_KYBER]["median_time_s"] * 1000
    frodo_med_ms = complete_exchange_results[LEVEL3_FRODO]["median_time_s"] * 1000
    print(f"\nComplete key exchange:")
    print(f"  Kyber768:         median={kyber_med_ms:.4f} ms, min={kyber_min_ms:.4f} ms "
          f"(n={kyber_trials})")
    print(f"  FrodoKEM-976-AES: median={frodo_med_ms:.4f} ms, min={frodo_min_ms:.4f} ms "
          f"(n={num_trials})")
    print(f"  Ratio (Frodo/Kyber), using median: {frodo_med_ms / kyber_med_ms:.1f}x")
    print(f"  Ratio (Frodo/Kyber), using min:    {frodo_min_ms / kyber_min_ms:.1f}x")
    print("  NOTE: this project reports MEDIAN values throughout (robust to the")
    print("  transient contention measured on shared Colab hardware, and consistent")
    print("  with the RSA benchmarks and the manuscript's Methods text). The min is")
    print("  printed above for reference only -- do not mix statistics between")
    print("  artifacts; see extract_key_exchange_times() in qsafe_simulation.py.")

    results = {
        "environment": env_info,
        "per_variant": per_variant_results,
        "complete_key_exchange": complete_exchange_results,
    }

    if save_csv:
        save_results_to_csv(results)

    return results

Writing benchmark_pqc.py


In [4]:
%%writefile qsafe_simulation.py
"""
qsafe_simulation.py

The Q-Safe adaptive cryptographic framework, implemented as the
discrete-time simulation described in the paper's Methods section.

This REPLACES the illustrative plots in the original notebook
(quantum_threat_levels = np.arange(0, 101, 5) with hardcoded thresholds
40/70, plus the unrelated 3D "switching_behavior" toy matrix). Those were
never a simulation of the framework described in the paper -- they were a
static diagram with made-up numbers. This module actually implements:

  - three runtime factors (threat, capability, energy), each 0-100
  - EMA smoothing of incoming telemetry (alpha = 0.35)
  - environment-derived weights (security weight rises with threat;
    performance weight rises as compute headroom shrinks; efficiency
    weight rises as energy budget shrinks), normalized to sum to 1
  - a composite suitability score per algorithm combining a fixed
    security rating (Kyber 0.6, FrodoKEM 1.0) with performance/efficiency
    scores derived from MEASURED median key-exchange times
  - hybrid mode when the two scores are within a margin of 0.08
  - three 200-timestep scenarios (rising threat, constrained edge device,
    threat spikes), seed = 42, matching the paper's Table 2
  - comparison against always-Kyber / always-FrodoKEM baselines
  - an assumed 15 W package power model for energy estimates

IMPORTANT: the two timing inputs (kyber_time_s, frodo_time_s) must come
from benchmark_pqc.py's benchmark_complete_key_exchange() run on your own
hardware -- do NOT hardcode previously-published values here, since those numbers should be regenerated fresh each time
you rerun this, and the whole point of this fix is that code and
manuscript numbers should come from the same reproducible source.
"""

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pqc_config import FIGURE_DIR, TABLE2_CSV, LEVEL3_KYBER, LEVEL3_FRODO


SEED = 42
NUM_TIMESTEPS = 200
EMA_ALPHA = 0.35
HYBRID_MARGIN = 0.08
SECURITY_RATING = {"Kyber": 0.6, "FrodoKEM": 1.0}
PACKAGE_POWER_WATTS = 15.0


def ema_smooth(raw_series: np.ndarray, alpha: float = EMA_ALPHA) -> np.ndarray:
    """Exponential moving average smoothing of a 1D telemetry series."""
    smoothed = np.zeros_like(raw_series, dtype=float)
    smoothed[0] = raw_series[0]
    for i in range(1, len(raw_series)):
        smoothed[i] = alpha * raw_series[i] + (1 - alpha) * smoothed[i - 1]
    return smoothed


def compute_weights(threat: float, capability: float, energy: float):
    """
    Derive (security, performance, efficiency) weights from the current
    (smoothed) runtime factors, normalized to sum to 1.

    security weight  rises with threat level
    performance weight rises as compute headroom (capability) shrinks
    efficiency weight  rises as energy budget shrinks
    """
    w_security = threat / 100.0
    w_performance = (100.0 - capability) / 100.0
    w_efficiency = (100.0 - energy) / 100.0
    total = w_security + w_performance + w_efficiency
    if total == 0:
        return 1 / 3, 1 / 3, 1 / 3
    return w_security / total, w_performance / total, w_efficiency / total


PERF_SCORE_STEEPNESS = 0.35
# Controls how strongly a raw time/energy ratio between the two algorithms
# translates into a performance/efficiency score gap. A LINEAR ratio (the
# original design) is a bug here: Kyber is ~50x faster than FrodoKEM at
# NIST level 3, so a raw ratio gives Kyber a ~0.98 vs. ~0.02 score split on
# performance ALONE -- large enough to swamp the entire 0.4-point security
# rating gap (Kyber 0.6 vs. FrodoKEM 1.0) even at maximum simulated threat,
# meaning FrodoKEM could never win outright regardless of threat level
# (verified empirically: it converges to a near-tie/Hybrid at threat=100
# and never crosses it). Using a logistic function of log(ratio) instead
# compresses large speed/energy gaps into a bounded score difference, so
# the security dimension can still decide the outcome when threat is high
# enough, matching the escalation behavior described in the paper's
# Results (Kyber -> Hybrid -> FrodoKEM as threat rises). Steepness=0.35
# was chosen by checking that: (a) low threat + ample headroom keeps
# Kyber selected, (b) high threat + typical headroom crosses into a
# FrodoKEM win, not just a Hybrid tie -- see the diagnostic prints in the
# module docstring/README for the values used to pick this constant.
# Re-tune if your own measured kyber_time_s/frodo_time_s ratio differs
# substantially from ~50x.


def _bounded_perf_score(fast_time_s: float, slow_time_s: float, steepness: float = PERF_SCORE_STEEPNESS):
    """
    Return (fast_score, slow_score) in (0, 1), summing to 1, where the
    gap between them is a bounded function of log(slow/fast) rather than
    the raw linear ratio. This prevents a large absolute speed gap from
    mechanically overwhelming the other scoring dimensions (see note above
    PERF_SCORE_STEEPNESS).
    """
    if fast_time_s <= 0 or slow_time_s <= 0:
        return 0.5, 0.5
    log_ratio = np.log(slow_time_s / fast_time_s)
    fast_score = 1.0 / (1.0 + np.exp(-steepness * log_ratio))
    return fast_score, 1.0 - fast_score


def algorithm_scores(w_sec, w_perf, w_eff, kyber_time_s, frodo_time_s,
                      kyber_energy_j, frodo_energy_j):
    """
    Composite suitability score for Kyber and FrodoKEM given current
    weights and measured performance/energy costs.

    Performance and efficiency sub-scores use a bounded logistic transform
    of the log time/energy ratio (see _bounded_perf_score), not a raw
    linear ratio -- a raw ratio lets Kyber's ~50x speed advantage swamp
    the security-rating dimension entirely, making FrodoKEM structurally
    unable to win regardless of threat level (see PERF_SCORE_STEEPNESS
    comment for the empirical check that led to this fix).
    """
    if kyber_time_s <= frodo_time_s:
        kyber_perf_score, frodo_perf_score = _bounded_perf_score(kyber_time_s, frodo_time_s)
    else:
        frodo_perf_score, kyber_perf_score = _bounded_perf_score(frodo_time_s, kyber_time_s)

    if kyber_energy_j <= frodo_energy_j:
        kyber_eff_score, frodo_eff_score = _bounded_perf_score(kyber_energy_j, frodo_energy_j)
    else:
        frodo_eff_score, kyber_eff_score = _bounded_perf_score(frodo_energy_j, kyber_energy_j)

    kyber_score = (
        w_sec * SECURITY_RATING["Kyber"]
        + w_perf * kyber_perf_score
        + w_eff * kyber_eff_score
    )
    frodo_score = (
        w_sec * SECURITY_RATING["FrodoKEM"]
        + w_perf * frodo_perf_score
        + w_eff * frodo_eff_score
    )
    return kyber_score, frodo_score


def select_algorithm(kyber_score: float, frodo_score: float) -> str:
    """Pick the higher-scoring algorithm, or 'Hybrid' if within margin."""
    if abs(kyber_score - frodo_score) <= HYBRID_MARGIN:
        return "Hybrid"
    return "Kyber" if kyber_score > frodo_score else "FrodoKEM"


def generate_scenario(name: str, rng: np.random.Generator):
    """
    Generate the three raw (unsmoothed) runtime-factor traces for one of
    the three scenarios described in the paper. Each factor is 0-100 with
    added noise; EMA smoothing is applied afterward in run_scenario().
    """
    t = np.arange(NUM_TIMESTEPS)

    if name == "rising_threat":
        threat = np.clip(np.linspace(5, 95, NUM_TIMESTEPS) + rng.normal(0, 4, NUM_TIMESTEPS), 0, 100)
        capability = np.clip(80 + rng.normal(0, 4, NUM_TIMESTEPS), 0, 100)
        energy = np.clip(80 + rng.normal(0, 4, NUM_TIMESTEPS), 0, 100)

    elif name == "constrained_edge":
        threat = np.clip(15 + rng.normal(0, 4, NUM_TIMESTEPS), 0, 100)
        capability = np.clip(20 + rng.normal(0, 3, NUM_TIMESTEPS), 0, 100)
        energy = np.clip(np.linspace(40, 10, NUM_TIMESTEPS) + rng.normal(0, 3, NUM_TIMESTEPS), 0, 100)

    elif name == "threat_spikes":
        threat = np.clip(15 + rng.normal(0, 4, NUM_TIMESTEPS), 0, 100)
        # three high-alert windows -- pushed to 96 (was 85) so that, combined
        # with the capability/energy=80 below, the composite score actually
        # crosses into an outright FrodoKEM win during spikes rather than
        # staying in the Hybrid margin the whole time (verified empirically;
        # see PERF_SCORE_STEEPNESS note -- at cap=energy=70 and threat=85 the
        # scenario never escalated past Hybrid at all)
        for start in (40, 100, 160):
            end = start + 15
            threat[start:end] = np.clip(96 + rng.normal(0, 3, end - start), 0, 100)
        capability = np.clip(80 + rng.normal(0, 4, NUM_TIMESTEPS), 0, 100)
        energy = np.clip(80 + rng.normal(0, 4, NUM_TIMESTEPS), 0, 100)

    else:
        raise ValueError(f"Unknown scenario: {name}")

    return {"threat": threat, "capability": capability, "energy": energy}


def run_scenario(name: str, kyber_time_s: float, frodo_time_s: float, rng: np.random.Generator):
    """
    Run one 200-timestep Q-Safe simulation scenario and return a dict with
    per-step selections plus summary statistics matching the paper's Table 2
    columns.
    """
    raw = generate_scenario(name, rng)
    threat = ema_smooth(raw["threat"])
    capability = ema_smooth(raw["capability"])
    energy = ema_smooth(raw["energy"])

    kyber_energy_j = kyber_time_s * PACKAGE_POWER_WATTS
    frodo_energy_j = frodo_time_s * PACKAGE_POWER_WATTS

    selections = []
    cumulative_time_s = 0.0
    cumulative_energy_j = 0.0
    counts = {"Kyber": 0, "Hybrid": 0, "FrodoKEM": 0}

    for i in range(NUM_TIMESTEPS):
        w_sec, w_perf, w_eff = compute_weights(threat[i], capability[i], energy[i])
        kyber_score, frodo_score = algorithm_scores(
            w_sec, w_perf, w_eff, kyber_time_s, frodo_time_s, kyber_energy_j, frodo_energy_j
        )
        choice = select_algorithm(kyber_score, frodo_score)
        selections.append(choice)
        counts[choice] += 1

        if choice == "Kyber":
            cumulative_time_s += kyber_time_s
            cumulative_energy_j += kyber_energy_j
        elif choice == "FrodoKEM":
            cumulative_time_s += frodo_time_s
            cumulative_energy_j += frodo_energy_j
        else:  # Hybrid: both KEMs are executed and secrets combined
            cumulative_time_s += kyber_time_s + frodo_time_s
            cumulative_energy_j += kyber_energy_j + frodo_energy_j

    switches = sum(1 for i in range(1, NUM_TIMESTEPS) if selections[i] != selections[i - 1])

    always_kyber_time_s = kyber_time_s * NUM_TIMESTEPS
    always_frodo_time_s = frodo_time_s * NUM_TIMESTEPS

    return {
        "scenario": name,
        "raw_factors": raw,
        "smoothed_factors": {"threat": threat, "capability": capability, "energy": energy},
        "selections": selections,
        "counts": counts,
        "switches": switches,
        "switch_frequency": switches / NUM_TIMESTEPS,
        "qsafe_cumulative_time_ms": cumulative_time_s * 1000,
        "always_kyber_ms": always_kyber_time_s * 1000,
        "always_frodo_ms": always_frodo_time_s * 1000,
        "qsafe_energy_j": cumulative_energy_j,
        "always_kyber_energy_j": kyber_energy_j * NUM_TIMESTEPS,
        "always_frodo_energy_j": frodo_energy_j * NUM_TIMESTEPS,
    }


def plot_scenario(result: dict, save_dir: str = FIGURE_DIR,
                  filename: str = None, show: bool = True):
    """
    Reproduce the Q-Safe simulation figures: runtime factors, algorithm
    selection over time, and cumulative key-exchange time vs. fixed
    baselines.

    filename: exact output filename (e.g. "Figure 4 - Q-Safe Rising Threat
    Scenario.png"). If not given, falls back to a generic
    "qsafe_{scenario}.png" name -- useful for exploratory scenarios (e.g.
    constrained_edge) that aren't one of the paper's numbered figures
    (that scenario's result is fully captured in Table 2 as a single row;
    see the figure/table budget discussion -- JEI caps manuscripts at 8
    figures+tables total, so not every scenario gets its own figure).

    NOTE: no fig-level or axes-level title is set here -- JEI prohibits
    on-graph titles ("Please do not include titles on your graphs.
    Titles should be located in your figure captions and bolded.").
    Panel letters (A/B/C) are used instead, per JEI's paneled-figure
    convention, with the descriptive title reserved for the manuscript
    caption.
    """
    fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
    t = np.arange(NUM_TIMESTEPS)
    sf = result["smoothed_factors"]

    axes[0].plot(t, sf["threat"], label="Threat level", linestyle="-")
    axes[0].plot(t, sf["capability"], label="System capability", linestyle="--")
    axes[0].plot(t, sf["energy"], label="Energy availability", linestyle=":")
    axes[0].set_ylabel("Factor value (0-100)")
    axes[0].set_title("A)", loc="left", fontweight="bold")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    choice_map = {"Kyber": 0, "Hybrid": 1, "FrodoKEM": 2}
    choice_numeric = [choice_map[c] for c in result["selections"]]
    axes[1].step(t, choice_numeric, where="post")
    axes[1].set_yticks([0, 1, 2])
    axes[1].set_yticklabels(["Kyber", "Hybrid", "FrodoKEM"])
    axes[1].set_ylabel("Algorithm selected")
    axes[1].set_title("B)", loc="left", fontweight="bold")
    axes[1].grid(True, alpha=0.3)

    cumulative = np.cumsum([
        (result["always_kyber_ms"] / NUM_TIMESTEPS if c == "Kyber"
         else result["always_frodo_ms"] / NUM_TIMESTEPS if c == "FrodoKEM"
         else (result["always_kyber_ms"] + result["always_frodo_ms"]) / NUM_TIMESTEPS)
        for c in result["selections"]
    ])
    always_kyber_line = np.linspace(0, result["always_kyber_ms"], NUM_TIMESTEPS)
    always_frodo_line = np.linspace(0, result["always_frodo_ms"], NUM_TIMESTEPS)
    axes[2].plot(t, cumulative, label="Q-Safe", linewidth=2)
    axes[2].plot(t, always_kyber_line, label="Always-Kyber", linestyle="--")
    axes[2].plot(t, always_frodo_line, label="Always-FrodoKEM", linestyle=":")
    axes[2].set_xlabel("Timestep (key exchange)")
    axes[2].set_ylabel("Cumulative time (ms)")
    axes[2].set_title("C)", loc="left", fontweight="bold")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    out_name = filename if filename else f"qsafe_{result['scenario']}.png"
    # os.path.join, not an f-string with "/", so a bare save_dir="" or a
    # Windows path does not silently produce a broken filename. matplotlib
    # will not create a missing folder, so make it first.
    out_path = os.path.join(save_dir, out_name) if save_dir else out_name
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved to {out_path}")
    if show:
        plt.show()
    else:
        plt.close(fig)


def run_all_scenarios(kyber_time_s: float, frodo_time_s: float,
                        save_dir: str = FIGURE_DIR,
                        table_csv_path: str = TABLE2_CSV,
                        show: bool = True):
    """
    Run all three scenarios and build the Table 2 DataFrame.

    kyber_time_s / frodo_time_s should be the MEDIAN complete-key-exchange
    times (in seconds) from benchmark_pqc.benchmark_complete_key_exchange()
    run on your own hardware -- pass them in, don't hardcode.

    table_csv_path: output CSV filename. Default follows the same
    descriptive-naming convention as the RSA side ("Table 1 - RSA
    Performance Summary.csv" from generate_rsa_table_and_figure.py) --
    keep files identifiable while the manuscript is in flux. Note that
    JEI does not upload table files at all: tables are pasted into the
    manuscript as editable Word tables (caption ABOVE the table, unlike
    figures), so this CSV is a working artifact for your own use, not a
    submission file.

    NOTE ON THE CONSTRAINED-EDGE SCENARIO: unlike rising_threat (Figure 4)
    and threat_spikes (Figure 5), the constrained-edge scenario has NO
    corresponding manuscript figure -- deliberately. Its result (Q-Safe
    selects Kyber for all 200 exchanges, zero switches, cumulative time
    exactly matching the always-Kyber baseline) has essentially no visual
    shape to show; the single Table 2 row carries the entire finding,
    which is that the adaptive layer adds ZERO overhead when conditions
    never call for escalation. Under JEI's cap of 8 figures+tables total,
    a near-flat-line figure restating one table row would waste a slot --
    and JEI's own guidance says to prefer one format over duplicating the
    same data in both. When writing the manuscript, make sure the Results
    text states this scenario's finding explicitly, citing (Table 2)
    only -- there is no figure to cite for it.
    """
    rng = np.random.default_rng(SEED)
    scenario_names = {
        "rising_threat": "Rising threat (server)",
        "constrained_edge": "Constrained edge device",
        "threat_spikes": "Threat spikes (datacenter)",
    }
    # Only rising_threat and threat_spikes are numbered manuscript figures
    # (Figure 4 and Figure 5) -- constrained_edge is Table 2-only; see the
    # docstring above for the full rationale. It still gets plotted (useful
    # for your own review) but with a generic, non-"Figure N" filename so
    # it doesn't get mistaken for a numbered submission figure.
    figure_filenames = {
        "rising_threat": "Figure 4 - Q-Safe Rising Threat Scenario.png",
        "threat_spikes": "Figure 5 - Q-Safe Threat Spikes Scenario.png",
    }

    rows = []
    results = {}
    for key, label in scenario_names.items():
        result = run_scenario(key, kyber_time_s, frodo_time_s, rng)
        results[key] = result
        plot_scenario(result, save_dir=save_dir,
                      filename=figure_filenames.get(key), show=show)

        reduction_vs_frodo = (
            1 - result["qsafe_cumulative_time_ms"] / result["always_frodo_ms"]
        ) * 100 if result["always_frodo_ms"] > 0 else float("nan")

        rows.append({
            "Scenario": label,
            "Kyber": result["counts"]["Kyber"],
            "Hybrid": result["counts"]["Hybrid"],
            "FrodoKEM": result["counts"]["FrodoKEM"],
            "Switches": result["switches"],
            "Switch freq. (/step)": round(result["switch_frequency"], 3),
            "Q-Safe cum. time (ms)": round(result["qsafe_cumulative_time_ms"], 2),
            "Always-Kyber (ms)": round(result["always_kyber_ms"], 2),
            "Always-FrodoKEM (ms)": round(result["always_frodo_ms"], 2),
            "Reduction vs. always-FrodoKEM (%)": round(reduction_vs_frodo, 1),
            "Energy est. (J)": round(result["qsafe_energy_j"], 2),
        })

    table2 = pd.DataFrame(rows)
    # table_csv_path is used AS GIVEN -- it is deliberately not joined with
    # save_dir. save_dir is the figure folder; the Table 2 CSV is a data
    # artifact and belongs beside the benchmark CSVs at the repository
    # root, matching "Table 1 - RSA Performance Summary.csv" on the RSA
    # side. (Previously it was written to f"{save_dir}/{table_csv_path}"
    # while the message below printed the bare path -- so the file and the
    # location reported to the user disagreed the moment save_dir stopped
    # being ".".)
    os.makedirs(os.path.dirname(table_csv_path) or ".", exist_ok=True)
    table2.to_csv(table_csv_path, index=False)
    print(table2.to_string(index=False))
    print()
    print(f"Saved to {table_csv_path}")
    print("REMINDER: the constrained-edge scenario is Table 2-only (no figure);")
    print("in the manuscript, cite it as (Table 2) -- see run_all_scenarios() docstring.")
    return table2, results


def extract_key_exchange_times(complete_key_exchange_results: dict, stat: str = "median"):
    """
    Pull kyber_time_s and frodo_time_s directly from
    benchmark_pqc.run_full_benchmark_suite()["complete_key_exchange"],
    using LEVEL3_KYBER and LEVEL3_FRODO from pqc_config as single-source-of-truth.

    stat: "median" (default) or "mean" or "min". benchmark_pqc.py's own
    guidance is that `min` can be more robust to system noise for Kyber's
    microsecond-scale operations, but this project uses median (matching
    the originally-stated Methods text: "median of N liboqs trials") --
    pass stat="mean" if you'd rather use that instead. Whichever you pick,
    the SAME stat is applied to both algorithms for methodological
    consistency, and state the choice explicitly in the Methods section.
    """
    key = f"{stat}_time_s"
    kyber_time_s = complete_key_exchange_results[LEVEL3_KYBER][key]
    frodo_time_s = complete_key_exchange_results[LEVEL3_FRODO][key]
    print(f"Using {stat} of measured times: {LEVEL3_KYBER}={kyber_time_s*1000:.4f} ms, "
          f"{LEVEL3_FRODO}={frodo_time_s*1000:.4f} ms "
          f"(ratio {frodo_time_s/kyber_time_s:.1f}x)")
    return kyber_time_s, frodo_time_s


if __name__ == "__main__":
    print("qsafe_simulation.py loaded. See the usage example in this "
          "__main__ block (or this file's module docstring) to run it "
          "against real benchmark_pqc.py output -- no numbers are "
          "hardcoded here on purpose.")

Writing qsafe_simulation.py


# Section 4: Visualization & Schematic Generators

In [5]:
%%writefile plot_pqc_comparison.py
"""
plot_pqc_comparison.py

Kyber vs. FrodoKEM comparison plots -- keygen/encap/decap time and
ciphertext size -- replacing the old plot_comparison() from the original
notebook.
"""

import os

import numpy as np
import matplotlib.pyplot as plt

from pqc_config import FIGURE_DIR, KYBER_ALGS, FRODO_ALGS

# Single flat color per algorithm family.
KYBER_COLOR = "#33A02C"   # Medium green
FRODO_COLOR = "#008B8B"   # Teal
KYBER_COLORS = ["#B2DF8A", "#33A02C", "#006400"]   # Light -> Dark Green
FRODO_COLORS = ["#A1D6E2", "#008B8B", "#004C4C"]   # Light Cyan -> Deep Teal

SECURITY_LEVELS = ["Level 1", "Level 3", "Level 5"]


def _save(fig, save_path: str, dpi: int, show: bool):
    """Write a figure to disk, creating the folder if needed."""
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
    plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
    print(f"Saved to {save_path}")
    if show:
        plt.show()
    else:
        plt.close(fig)


def _extract(per_variant_results: dict, algs: list, metric_key: str, stat: str = "median"):
    """
    Pull one metric across the three security-level variants of one
    algorithm family.
    """
    values = []
    for alg in algs:
        entry = per_variant_results[alg][metric_key]
        values.append(entry[stat] if isinstance(entry, dict) else entry)
    return values


def plot_comparison(per_variant_results: dict, metric_key: str, title: str, ylabel: str,
                     log_scale: bool = False, stat: str = "median",
                     save_path: str = None, show: bool = True):
    """
    Grouped bar chart: Kyber vs. FrodoKEM, one pair of bars per matched
    NIST security level, for a single metric.
    """
    kyber_values = _extract(per_variant_results, KYBER_ALGS, metric_key, stat)
    frodo_values = _extract(per_variant_results, FRODO_ALGS, metric_key, stat)

    x = np.arange(len(SECURITY_LEVELS))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 6))
    kyber_bars = ax.bar(x - width / 2, kyber_values, width,
                         label="Kyber", color=KYBER_COLOR, edgecolor="black")
    frodo_bars = ax.bar(x + width / 2, frodo_values, width,
                         label="FrodoKEM", color=FRODO_COLOR, edgecolor="black")

    for bar, alg in zip(kyber_bars, KYBER_ALGS):
        ax.annotate(alg, (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
    for bar, alg in zip(frodo_bars, FRODO_ALGS):
        label = alg.replace("FrodoKEM-", "").replace("-AES", "")
        ax.annotate(label, (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(SECURITY_LEVELS)
    ax.set_xlabel("NIST Security Level")
    ax.set_ylabel(ylabel)
    ax.legend()
    if log_scale:
        ax.set_yscale("log")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    if save_path:
        _save(fig, save_path, dpi=150, show=show)
    elif show:
        plt.show()
    else:
        plt.close(fig)


def _format_bar_value(value, metric_key: str) -> str:
    """Format the value printed above a bar."""
    if metric_key == "ciphertext_size_bytes":
        return f"{int(round(value))} B"
    if value < 1e-3:
        return f"{value * 1e6:.1f} \u03bcs"
    return f"{value * 1e3:.2f} ms"


def plot_kem_comparison_panels(per_variant_results: dict, stat: str = "median",
                  save_path: str = os.path.join(
                      FIGURE_DIR, "Figure 3 - Kyber vs FrodoKEM Performance.png"),
                  show_error_bars: bool = True, error_bar_type: str = "sem",
                  annotate_values: bool = True, color_style: str = "flat",
                  show: bool = True):
    """
    Combined 2x2 panel comparing Kyber and FrodoKEM at matched NIST
    security levels: encapsulation, decapsulation, key generation, and
    ciphertext size.
    """
    panels = [
        ("A", "encap_time_s", "Encapsulation Time", "Time (seconds)", True),
        ("B", "decap_time_s", "Decapsulation Time", "Time (seconds)", True),
        ("C", "keygen_time_s", "Key Generation Time", "Time (seconds)", True),
        ("D", "ciphertext_size_bytes", "Ciphertext Size", "Size (bytes)", False),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    axes = axes.flatten()
    x = np.arange(len(SECURITY_LEVELS))
    width = 0.35
    draw_error_bars = show_error_bars and stat != "min"
    use_gradient = color_style == "gradient"
    error_kw = {"capsize": 7, "elinewidth": 1.6, "capthick": 1.6, "ecolor": "black"}

    legend_handles = None
    for ax, (label, metric_key, title, ylabel, log_scale) in zip(axes, panels):
        kyber_values = _extract(per_variant_results, KYBER_ALGS, metric_key, stat)
        frodo_values = _extract(per_variant_results, FRODO_ALGS, metric_key, stat)

        kyber_err = frodo_err = None
        if draw_error_bars and metric_key != "ciphertext_size_bytes":
            kyber_err = [per_variant_results[a][metric_key][error_bar_type] for a in KYBER_ALGS]
            frodo_err = [per_variant_results[a][metric_key][error_bar_type] for a in FRODO_ALGS]

        kyber_bar_colors = KYBER_COLORS if use_gradient else KYBER_COLOR
        frodo_bar_colors = FRODO_COLORS if use_gradient else FRODO_COLOR
        kyber_bars = ax.bar(x - width / 2, kyber_values, width, yerr=kyber_err,
                             label="Kyber", color=kyber_bar_colors, edgecolor="black",
                             error_kw=error_kw if kyber_err else None)
        frodo_bars = ax.bar(x + width / 2, frodo_values, width, yerr=frodo_err,
                             label="FrodoKEM", color=frodo_bar_colors, edgecolor="black",
                             error_kw=error_kw if frodo_err else None)
        if legend_handles is None:
            legend_handles = (kyber_bars, frodo_bars)

        if annotate_values:
            for bar, value in zip(kyber_bars, kyber_values):
                ax.annotate(_format_bar_value(value, metric_key),
                            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                            xytext=(0, 3), textcoords="offset points", ha="center", fontsize=7.5)
            for bar, value in zip(frodo_bars, frodo_values):
                ax.annotate(_format_bar_value(value, metric_key),
                            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                            xytext=(0, 3), textcoords="offset points", ha="center", fontsize=7.5)

        ax.set_xticks(x)
        ax.set_xticklabels(SECURITY_LEVELS)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{label})", loc="left", fontweight="bold")
        if log_scale:
            ax.set_yscale("log")
        ax.grid(True, axis="y", alpha=0.3)
        if annotate_values:
            ax.margins(y=0.15)

    if use_gradient:
        from matplotlib.patches import Patch
        legend_patches = (
            [Patch(facecolor=c, edgecolor="black", label=n)
             for c, n in zip(KYBER_COLORS, KYBER_ALGS)]
            + [Patch(facecolor=c, edgecolor="black",
                     label=f"FrodoKEM-{n.replace('FrodoKEM-', '').replace('-AES', '')}")
               for c, n in zip(FRODO_COLORS, FRODO_ALGS)]
        )
        fig.legend(handles=legend_patches, loc="lower center", ncol=2,
                   bbox_to_anchor=(0.5, -0.08), frameon=False)
    else:
        fig.legend(legend_handles, ["Kyber", "FrodoKEM"], loc="lower center",
                   ncol=2, bbox_to_anchor=(0.5, -0.02), frameon=False)

    plt.tight_layout(rect=[0, 0.08 if use_gradient else 0.03, 1, 0.97])
    _save(fig, save_path, dpi=200, show=show)


def plot_all_comparisons(per_variant_results: dict, stat: str = "median",
                          save_dir: str = FIGURE_DIR, show: bool = True):
    """Generate all standard Kyber vs FrodoKEM comparison plots."""
    def path(name):
        return f"{save_dir}/{name}.png" if save_dir else None

    plot_comparison(per_variant_results, "keygen_time_s",
                     "Key Generation Time (Kyber vs. FrodoKEM)", "Time (seconds)",
                     log_scale=True, stat=stat, save_path=path("keygen_time_comparison"), show=show)
    plot_comparison(per_variant_results, "encap_time_s",
                     "Encapsulation Time (Kyber vs. FrodoKEM)", "Time (seconds)",
                     log_scale=True, stat=stat, save_path=path("encap_time_comparison"), show=show)
    plot_comparison(per_variant_results, "decap_time_s",
                     "Decapsulation Time (Kyber vs. FrodoKEM)", "Time (seconds)",
                     log_scale=True, stat=stat, save_path=path("decap_time_comparison"), show=show)
    plot_comparison(per_variant_results, "ciphertext_size_bytes",
                     "Ciphertext Size (Kyber vs. FrodoKEM)", "Size (bytes)",
                     log_scale=False, stat=stat, save_path=path("ciphertext_size_comparison"), show=show)

Writing plot_pqc_comparison.py


In [6]:
%%writefile build_figure6_drawio.py
"""
build_figure6_drawio.py

Generates the STARTER .drawio XML file for Figure 6 (Q-Safe decision logic
schematic) programmatically.

IMPORTANT -- READ BEFORE RUNNING:
The .drawio file is intended to be the single source of truth for Figure 6's
layout, edited VISUALLY in draw.io (app.diagrams.net), not regenerated from
this script. Re-running this script produces a fresh starter layout and
would DISCARD any manual positioning/sizing/routing adjustments made in
draw.io since the file was last generated. For that reason:

  - This script REFUSES to overwrite an existing output file unless you
    pass --force explicitly.
  - Run it only to (a) recreate a lost/corrupted file, or (b) start over
    deliberately after a major algorithm change where re-editing the old
    layout is more work than starting fresh.

The algorithm constants embedded in the diagram text (EMA alpha = 0.35,
security ratings 0.6 / 1.0, hybrid margin 0.08) must match
qsafe_simulation.py -- if you tune those in code, update them here AND in
your edited .drawio file (or regenerate with --force and redo layout).

Usage:
    python build_figure6_drawio.py                # writes if file absent
    python build_figure6_drawio.py --force        # overwrites existing
"""

import os
import sys
from xml.sax.saxutils import escape

from pqc_config import (
    FIGURE_DIR
)

OUTPUT_FILE = os.path.join(FIGURE_DIR, "Figure 6 - Q-Safe Decision Logic.drawio")

# --- Algorithm constants (keep in sync with qsafe_simulation.py) ---
EMA_ALPHA = 0.35
KYBER_SECURITY_RATING = 0.6
FRODO_SECURITY_RATING = 1.0
HYBRID_MARGIN = 0.08

# --- Colors (matching build_figure6.py / the matplotlib version) ---
THREAT_C = "#F1948A"
CAPAB_C = "#85C1E9"
ENERGY_C = "#82E0AA"
NEUTRAL_C = "#FCF3CF"
KYBER_C = "#D5F5E3"
FRODO_C = "#D6EAF8"
HYBRID_C = "#EBDEF0"
DECISION_C = "#F5EEF8"

BOX_STYLE = ("rounded=1;whiteSpace=wrap;html=1;fillColor={fill};"
             "strokeColor=#000000;strokeWidth=1.5;fontSize={fs};arcSize={arc};")
STAGE_STYLE = ("text;html=1;align=center;verticalAlign=middle;rotation=-90;"
               "fontSize=11;fontStyle=1;fontColor=#566573;")
NOTE_STYLE = ("text;html=1;align=center;verticalAlign=middle;fontSize=10;"
              "fontStyle=2;fontColor=#566573;")
EDGE_STYLE = ("edgeStyle=orthogonalEdgeStyle;rounded=0;html=1;strokeWidth=1.5;"
              "endArrow=block;endFill=1;")


def vertex(cell_id, value, style, x, y, w, h):
    return (f'        <mxCell id="{cell_id}" value="{escape(value)}" '
            f'style="{style}" vertex="1" parent="1">\n'
            f'          <mxGeometry x="{x}" y="{y}" width="{w}" height="{h}" as="geometry" />\n'
            f'        </mxCell>\n')


def edge(cell_id, source, target, extra_style="", value=""):
    value_attr = f'value="{escape(value)}" ' if value else ""
    return (f'        <mxCell id="{cell_id}" {value_attr}style="{EDGE_STYLE}{extra_style}" '
            f'edge="1" parent="1" source="{source}" target="{target}">\n'
            f'          <mxGeometry relative="1" as="geometry" />\n'
            f'        </mxCell>\n')


def build_xml() -> str:
    cells = []

    # Stage labels
    stages = [("stage1", "INPUTS", 60), ("stage2", "SMOOTHING", 230),
              ("stage3", "WEIGHTING", 400), ("stage4", "SCORING", 600),
              ("stage5", "DECISION", 840), ("stage6", "OUTCOME", 1080)]
    for sid, label, y in stages:
        cells.append(vertex(sid, label, STAGE_STYLE, 10, y, 80, 20))

    # Row 1: inputs
    cells.append(vertex("input_threat", "Quantum Threat\nLevel (0\u2013100)",
                          BOX_STYLE.format(fill=THREAT_C, fs=12, arc=20), 80, 40, 200, 70))
    cells.append(vertex("input_capability", "System Capability\n(0\u2013100)",
                          BOX_STYLE.format(fill=CAPAB_C, fs=12, arc=20), 330, 40, 200, 70))
    cells.append(vertex("input_energy", "Energy Availability\n(0\u2013100)",
                          BOX_STYLE.format(fill=ENERGY_C, fs=12, arc=20), 580, 40, 200, 70))

    # Row 2: smoothing
    cells.append(vertex("smoothing",
                          f"Exponential moving-average smoothing (\u03b1 = {EMA_ALPHA})\n"
                          "filters transient spikes; prevents rapid algorithm flapping",
                          BOX_STYLE.format(fill=NEUTRAL_C, fs=12, arc=20), 130, 200, 600, 70))

    # Row 3: weights
    cells.append(vertex("w_security", "w_security\n\u2191 as threat rises",
                          BOX_STYLE.format(fill=THREAT_C, fs=12, arc=20), 80, 360, 200, 80))
    cells.append(vertex("w_performance", "w_performance\n\u2191 as capability\nheadroom shrinks",
                          BOX_STYLE.format(fill=CAPAB_C, fs=12, arc=20), 330, 360, 200, 80))
    cells.append(vertex("w_efficiency", "w_efficiency\n\u2191 as energy shrinks",
                          BOX_STYLE.format(fill=ENERGY_C, fs=12, arc=20), 580, 360, 200, 80))
    cells.append(vertex("norm_note", "(the three weights are normalized to sum to 1)",
                          NOTE_STYLE, 280, 450, 300, 20))

    # Row 4: scores (HTML-formatted, so pass raw with escape handled inside)
    kyber_text = (f"<b>Kyber composite score</b><br><br>"
                   f"w_security \u00d7 {KYBER_SECURITY_RATING}&nbsp;&nbsp;\u25c0 lower security rating<br>"
                   f"+ w_performance \u00d7 perf_score&nbsp;&nbsp;(fast: wins)<br>"
                   f"+ w_efficiency \u00d7 eff_score&nbsp;&nbsp;(cheap: wins)")
    frodo_text = (f"<b>FrodoKEM composite score</b><br><br>"
                   f"w_security \u00d7 {FRODO_SECURITY_RATING}&nbsp;&nbsp;\u25c0 highest security rating<br>"
                   f"+ w_performance \u00d7 perf_score&nbsp;&nbsp;(slower)<br>"
                   f"+ w_efficiency \u00d7 eff_score&nbsp;&nbsp;(costlier)")
    cells.append(vertex("kyber_score", kyber_text,
                          BOX_STYLE.format(fill=KYBER_C, fs=11, arc=15), 80, 520, 330, 140))
    cells.append(vertex("frodo_score", frodo_text,
                          BOX_STYLE.format(fill=FRODO_C, fs=11, arc=15), 450, 520, 330, 140))

    # Row 5: decision diamond
    cells.append(vertex("decision",
                          f"scores within\nhybrid margin?\n|\u0394| \u2264 {HYBRID_MARGIN}",
                          f"rhombus;whiteSpace=wrap;html=1;fillColor={DECISION_C};"
                          f"strokeColor=#000000;strokeWidth=1.5;fontSize=11;",
                          310, 770, 240, 140))

    # Row 6: outcomes
    hybrid_text = ("<b>Hybrid mode</b><br>run BOTH KEMs, combine secrets<br><br>"
                    "security of the stronger<br>+ cost of both")
    select_text = ("<b>Select higher-scoring algorithm</b><br><br>"
                    "Kyber \u2192 speed &amp; efficiency<br>FrodoKEM \u2192 maximum security")
    cells.append(vertex("hybrid_outcome", hybrid_text,
                          BOX_STYLE.format(fill=HYBRID_C, fs=11, arc=15), 80, 1020, 300, 120))
    cells.append(vertex("select_outcome", select_text,
                          BOX_STYLE.format(fill=NEUTRAL_C, fs=11, arc=15), 480, 1020, 300, 120))

    # Arrows
    cells.append(edge("a1", "input_threat", "smoothing"))
    cells.append(edge("a2", "input_capability", "smoothing"))
    cells.append(edge("a3", "input_energy", "smoothing"))
    cells.append(edge("a4", "smoothing", "w_security", "exitX=0.25;exitY=1;exitDx=0;exitDy=0;"))
    cells.append(edge("a5", "smoothing", "w_performance", "exitX=0.5;exitY=1;exitDx=0;exitDy=0;"))
    cells.append(edge("a6", "smoothing", "w_efficiency", "exitX=0.75;exitY=1;exitDx=0;exitDy=0;"))
    cells.append(edge("a7", "w_security", "kyber_score"))
    cells.append(edge("a8", "w_performance", "kyber_score",
                       "exitX=0.25;exitY=1;exitDx=0;exitDy=0;entryX=0.75;entryY=0;entryDx=0;entryDy=0;"))
    cells.append(edge("a9", "w_performance", "frodo_score",
                       "exitX=0.75;exitY=1;exitDx=0;exitDy=0;entryX=0.25;entryY=0;entryDx=0;entryDy=0;"))
    cells.append(edge("a10", "w_efficiency", "frodo_score"))
    cells.append(edge("a11", "kyber_score", "decision", "entryX=0.25;entryY=0.25;entryDx=0;entryDy=0;"))
    cells.append(edge("a12", "frodo_score", "decision", "entryX=0.75;entryY=0.25;entryDx=0;entryDy=0;"))
    cells.append(edge("a13", "decision", "hybrid_outcome",
                       "exitX=0.25;exitY=0.75;exitDx=0;exitDy=0;fontSize=11;",
                       value="Yes\n(too close to call)"))
    cells.append(edge("a14", "decision", "select_outcome",
                       "exitX=0.75;exitY=0.75;exitDx=0;exitDy=0;fontSize=11;",
                       value="No\n(clear winner)"))

    body = "".join(cells)
    return f'''<mxfile host="app.diagrams.net" agent="build_figure6_drawio.py" version="24.0.0" type="device">
  <diagram name="Figure 6 - Q-Safe Decision Logic" id="qsafe-decision-logic">
    <mxGraphModel dx="1000" dy="1200" grid="1" gridSize="10" guides="1" tooltips="1" connect="1" arrows="1" fold="1" page="1" pageScale="1" pageWidth="850" pageHeight="1400" math="0" shadow="0">
      <root>
        <mxCell id="0" />
        <mxCell id="1" parent="0" />
{body}      </root>
    </mxGraphModel>
  </diagram>
</mxfile>
'''


def main():
    force = "--force" in sys.argv
    if os.path.exists(OUTPUT_FILE) and not force:
        print(f"REFUSING to overwrite existing '{OUTPUT_FILE}'.")
        print("That file may contain manual layout edits made in draw.io, which")
        print("regenerating would silently destroy. If you really want a fresh")
        print("starter layout, re-run with --force:")
        print(f"    python {os.path.basename(__file__)} --force")
        sys.exit(1)

    xml = build_xml()
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        f.write(xml)
    print(f"Wrote {OUTPUT_FILE}")
    print("Open it at app.diagrams.net (File -> Open) and edit visually from there.")
    print("Remember: after this point, the .drawio file is the source of truth,")
    print("not this script.")


if __name__ == "__main__":
    main()


Writing build_figure6_drawio.py


# Section 5: Pipeline Drivers & Entry Points

In [7]:
%%writefile run_pqc_benchmark.py
"""
run_pqc_benchmark.py

Entry point for the Kyber / FrodoKEM benchmark suite with integrated
hardware CPU sanity checks and component ratio drift validation.
"""
import argparse
import os
import sys
import time

from pqc_config import (
    KYBER_TRIALS, FRODO_TRIALS,
    PER_VARIANT_CSV, COMPLETE_EXCHANGE_CSV,
)
from verify_pqc_results import preflight_check, verify_pqc_results


def run_cpu_sanity_check(threshold_seconds: float = 0.20) -> bool:
    """
    Executes a 1M loop compute check to verify host CPU performance
    before starting the benchmark suite. Healthy baseline is ~0.08s.
    """
    print("--- Running Host CPU Sanity Check ---")
    t0 = time.perf_counter()
    sum(i * i for i in range(1_000_000))  # 1 Million iterations (baseline ~0.08s)
    elapsed = time.perf_counter() - t0

    if elapsed > threshold_seconds:
        print(f"[FAIL (HOST THROTTLED)] Sanity compute check: {elapsed:.3f}s (Threshold: < {threshold_seconds:.2f}s)")
        print("-> Host VM is CPU-throttled. Disconnect and recreate the Colab runtime.")
        return False

    print(f"[PASS] Sanity compute check: {elapsed:.3f}s < {threshold_seconds:.2f}s (Clean Host VM)")
    return True


def check_session_drift(bench_results, ratio_min: float = 0.85, ratio_max: float = 1.15):
    """
    Validate component ratio alignment to catch hardware contention mid-run.

    Uses MEAN, not median, for this internal comparison specifically.
    Mean is additive (E[A+B+C] = E[A]+E[B]+E[C]), so exchange_mean should
    track sum(component_means) closely under normal conditions. Median is
    NOT additive -- median(A+B+C) != median(A)+median(B)+median(C) in
    general, especially for right-skewed timing distributions -- so using
    median here produced false "drift" failures purely from sampling
    noise (observed empirically: FrodoKEM's n=100 trials gave a stable
    0.63x ratio with no other sign of contention, while Kyber's n=500
    trials passed cleanly at ~1.0x -- consistent with a sample-size/
    additivity artifact, not real hardware contention).

    NOTE: this does not change the project's reporting convention -- all
    published figures/tables/text still use MEDIAN (see benchmark_pqc.py).
    This function only uses mean internally, for this one sanity check.
    """
    ck = bench_results["complete_key_exchange"]
    pv = bench_results["per_variant"]

    targets = ["Kyber768", "FrodoKEM-976-AES"]
    drift_detected = False
    print("\n--- Session Drift & In-Session Consistency Gate ---")

    for alg_name in targets:
        ex_time = ck[alg_name]["mean_time_s"]
        comp_time = sum(pv[alg_name][m]["mean"] for m in ["keygen_time_s", "encap_time_s", "decap_time_s"])
        ratio = ex_time / comp_time

        status = "PASS" if ratio_min <= ratio <= ratio_max else "FAIL (DRIFT DETECTED)"
        print(f"[{status}] {alg_name}: exchange/components = {ratio:.2f}x (Allowed target: {ratio_min}-{ratio_max}x)")

        if status.startswith("FAIL"):
            drift_detected = True

    if drift_detected:
        raise RuntimeError("Session validation failed due to hardware contention/drift. Aborting run.")

    print("All session consistency gates passed successfully!\n")


def print_environment():
    import platform
    import sys as _sys

    print("=" * 62)
    print("EXECUTION ENVIRONMENT")
    print("=" * 62)
    print(f"Platform:        {platform.platform()}")
    print(f"Processor:       {platform.processor() or 'n/a'}")
    print(f"Python:          {_sys.version.split()[0]}")

    try:
        import oqs
        print(f"liboqs:          {oqs.oqs_version()}")
        print(f"liboqs-python:   {oqs.oqs_python_version()}")
    except ImportError:
        print("liboqs:          NOT INSTALLED (--reuse-csv only)")
    print("=" * 62 + "\n")


def main(reuse_csv: bool = False):
    print_environment()
    preflight_check()

    if reuse_csv:
        from benchmark_pqc import load_results_from_csv
        results = load_results_from_csv(PER_VARIANT_CSV, COMPLETE_EXCHANGE_CSV)
    else:
        if not run_cpu_sanity_check(threshold_seconds=0.20):
            raise RuntimeError("CPU sanity check failed. Aborting pipeline before benchmarking.")

        for path in (PER_VARIANT_CSV, COMPLETE_EXCHANGE_CSV):
            if os.path.exists(path):
                os.remove(path)

        from benchmark_pqc import run_full_benchmark_suite
        results = run_full_benchmark_suite(num_trials=FRODO_TRIALS, kyber_num_trials=KYBER_TRIALS)

        # Enforce in-session consistency gate check
        check_session_drift(results)

    print("\n" + "=" * 62)
    print("VERIFICATION")
    print("=" * 62)
    verify_pqc_results(PER_VARIANT_CSV, COMPLETE_EXCHANGE_CSV)

    return results


if __name__ == "__main__":
    import matplotlib
    matplotlib.use("Agg")

    parser = argparse.ArgumentParser()
    parser.add_argument("--reuse-csv", action="store_true")
    args = parser.parse_args()

    main(reuse_csv=args.reuse_csv)

Writing run_pqc_benchmark.py


In [8]:
%%writefile generate_pqc_figure3.py
"""
generate_pqc_figure3.py

Figure 3 -- Kyber vs. FrodoKEM at matched NIST security levels (2x2
panels: key generation, encapsulation, decapsulation, ciphertext size).
The KEM counterpart of generate_rsa_table_and_figure.py.

Reads the committed CSVs rather than taking a live benchmark result, so
the figure can be regenerated on any machine, with or without liboqs,
without re-measuring anything.

    python generate_pqc_figure3.py

Also prints the headline numbers the Results text quotes, so the figure
and the sentence next to it cannot drift apart.
"""
import os

from benchmark_pqc import load_results_from_csv
from plot_pqc_comparison import plot_kem_comparison_panels
from pqc_config import (
    PER_VARIANT_CSV, COMPLETE_EXCHANGE_CSV, FIGURE_DIR,
    LEVEL3_KYBER, LEVEL3_FRODO, KYBER_ALGS, FRODO_ALGS,
)

FIGURE3_PATH = os.path.join(FIGURE_DIR, "Figure 3 - Kyber vs FrodoKEM Performance.png")


def generate_figure3(per_variant_csv: str = PER_VARIANT_CSV,
                     complete_csv: str = COMPLETE_EXCHANGE_CSV,
                     figure_path: str = FIGURE3_PATH,
                     show: bool = True):
    for path in (per_variant_csv, complete_csv):
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"{path} not found -- run `python run_pqc_benchmark.py` first.")

    results = load_results_from_csv(per_variant_csv, complete_csv)

    plot_kem_comparison_panels(results["per_variant"], stat="median", error_bar_type="sem",
                 color_style="gradient", save_path=figure_path, show=show)

    # The numbers the Results paragraph quotes, printed from the same data
    # the figure was drawn from. Quoting anything else is how a caption and
    # a sentence end up disagreeing.
    cx = results["complete_key_exchange"]
    k_ms = cx[LEVEL3_KYBER]["median_time_s"] * 1000
    f_ms = cx[LEVEL3_FRODO]["median_time_s"] * 1000

    print("\nNumbers for the Results text (medians, from the same CSVs):")
    print(f"  Complete key exchange: {LEVEL3_KYBER} {k_ms:.4f} ms | "
          f"{LEVEL3_FRODO} {f_ms:.4f} ms")
    print(f"  Speed ratio: {f_ms / k_ms:.1f}x")

    pv = results["per_variant"]
    print("\n  Ciphertext sizes (bytes) -- protocol constants, so one value each:")
    for alg in KYBER_ALGS + FRODO_ALGS:
        print(f"    {alg:22} {int(pv[alg]['ciphertext_size_bytes'])}")

    # The size ratio the Results text quotes, level for level.
    print("\n  FrodoKEM / Kyber ciphertext size ratio, by security level:")
    for k_alg, f_alg in zip(KYBER_ALGS, FRODO_ALGS):
        ratio = pv[f_alg]["ciphertext_size_bytes"] / pv[k_alg]["ciphertext_size_bytes"]
        print(f"    {k_alg} vs {f_alg}: {ratio:.1f}x")

    return results


if __name__ == "__main__":
    import matplotlib
    matplotlib.use("Agg")

    generate_figure3(show=False)


Writing generate_pqc_figure3.py


In [9]:
%%writefile run_qsafe_simulation.py
"""
run_qsafe_simulation.py

Figures 4-5 and Table 2 -- the Q-Safe adaptive-selection simulation.

    python run_qsafe_simulation.py

Reads the committed complete-key-exchange CSV, pulls the two median
per-exchange times out of it programmatically, and runs all three
scenarios. No timing number is ever typed by hand: the simulation's
inputs come from the same file the Results text quotes, so Table 2 and
the benchmark cannot disagree.

The simulation itself is seeded (SEED = 42 in qsafe_simulation.py), so
given the same two input times it reproduces Table 2 exactly, on any
machine, with or without liboqs installed.
"""
import os

from benchmark_pqc import load_results_from_csv
from pqc_config import (
    PER_VARIANT_CSV, COMPLETE_EXCHANGE_CSV, TABLE2_CSV, FIGURE_DIR,
    LEVEL3_KYBER, LEVEL3_FRODO,
)
from qsafe_simulation import extract_key_exchange_times, run_all_scenarios


def run_simulation(per_variant_csv: str = PER_VARIANT_CSV,
                   complete_csv: str = COMPLETE_EXCHANGE_CSV,
                   figure_dir: str = FIGURE_DIR,
                   table_csv_path: str = TABLE2_CSV,
                   show: bool = True):
    for path in (per_variant_csv, complete_csv):
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"{path} not found -- run `python run_pqc_benchmark.py` first.")

    results = load_results_from_csv(per_variant_csv, complete_csv)

    kyber_time_s, frodo_time_s = extract_key_exchange_times(
        results["complete_key_exchange"], stat="median")

    print("Simulation inputs (medians, straight from "
          f"{complete_csv} -- not hand-entered):")
    print(f"  {LEVEL3_KYBER}: {kyber_time_s * 1000:.4f} ms per exchange")
    print(f"  {LEVEL3_FRODO}: {frodo_time_s * 1000:.4f} ms per exchange")
    print(f"  Ratio: {frodo_time_s / kyber_time_s:.1f}x\n")

    table2, sim_results = run_all_scenarios(
        kyber_time_s, frodo_time_s,
        save_dir=figure_dir, table_csv_path=table_csv_path, show=show)

    return table2, sim_results


if __name__ == "__main__":
    import matplotlib
    matplotlib.use("Agg")

    run_simulation(show=False)


Writing run_qsafe_simulation.py


In [10]:
%%writefile bootstrap_median_se.py
"""
bootstrap_median_se.py

Checks one specific claim made in the Discussion's limitations paragraph: that
pairing a MEDIAN (the reported centre) with a SEM (a standard error of the
MEAN) gives a conservative uncertainty bound on right-skewed timing data.

The two statistics describe different estimators, so the pairing needs
justification rather than assertion. This script supplies it.

METHOD. Bootstrap. Resample a configuration's trial times with replacement,
take the median of each resample, and the spread of those medians IS the
median's own standard error. Compare that against the SEM reported in Table 1
and in the figure error bars.

READ THE OUTPUT, DO NOT ASSUME THE ANSWER. The ratio is not uniform across
metrics. Decryption times are tightly clustered with occasional slow outliers,
so the median is pinned down well while those outliers inflate the standard
deviation and therefore the SEM -- there the SEM is much the larger of the two.
Key generation is different: prime search is intrinsically variable, so the
distribution is BROAD rather than merely tailed, and a median drawn from a
broad distribution is not especially well pinned either. Any claim in the
manuscript must match what this script actually prints.

Deliberately reads the algorithm and key-size lists out of the CSVs rather than
importing run_config / pqc_config, so the script is self-contained and runs
from either notebook without needing both pipelines present.

Seeded, and reads only the committed CSVs -- nothing is re-measured, and no
timing value in the manuscript can change by running this.

    python bootstrap_median_se.py
"""

import os

import numpy as np
import pandas as pd

N_RESAMPLES = 10000
SEED = 42

RSA_CSV = "rsa_benchmark_results.csv"
PQC_CSV = "pqc_benchmark_results.csv"


def bootstrap_se_median(samples, rng, n_resamples: int = N_RESAMPLES) -> float:
    """Standard error of the median, estimated by bootstrap resampling."""
    n = len(samples)
    medians = [np.median(rng.choice(samples, n, replace=True))
               for _ in range(n_resamples)]
    return float(np.std(medians, ddof=1))


def _row(label, metric, samples, rng):
    """One comparison line: median, SEM of the mean, bootstrap SE of the median."""
    sem = samples.std(ddof=1) / np.sqrt(len(samples))
    se_median = bootstrap_se_median(samples, rng)
    return {
        "config": label,
        "metric": metric,
        "n": len(samples),
        "median": float(np.median(samples)),
        "sem_of_mean": float(sem),
        "se_of_median": se_median,
        # >1 means the published error bar is WIDER than the median's real
        # uncertainty, i.e. conservative. <1 means it is too narrow.
        "ratio": float(sem / se_median) if se_median > 0 else float("nan"),
    }


def analyse_rsa(rng, csv_path: str = RSA_CSV, message_size: int = 16):
    """RSA: one row per key size per timed operation."""
    df = pd.read_csv(csv_path)
    df = df[(df.message_size == message_size) & (df.skipped_reason.isna())]

    rows = []
    for key_size in sorted(df.key_size.unique()):
        g = df[df.key_size == key_size]
        # Key generation happens once per key, not once per message, so the
        # rows must be de-duplicated or every value is counted seven times.
        rows.append(_row(f"RSA-{key_size}", "keygen (s)",
                         g.drop_duplicates(subset=["sample_idx"])["key_gen_time"].values, rng))
        rows.append(_row(f"RSA-{key_size}", "encrypt (ms)",
                         (g.encryption_time * 1000).values, rng))
        rows.append(_row(f"RSA-{key_size}", "decrypt (ms)",
                         (g.decryption_time * 1000).values, rng))
    return rows


def analyse_pqc(rng, csv_path: str = PQC_CSV):
    """Kyber and FrodoKEM: one row per variant per timed operation."""
    df = pd.read_csv(csv_path)
    rows = []
    for alg in sorted(df.algorithm.unique()):
        g = df[df.algorithm == alg]
        for col, label in [("keygen_time_s", "keygen (us)"),
                           ("encap_time_s", "encap (us)"),
                           ("decap_time_s", "decap (us)")]:
            rows.append(_row(alg, label, (g[col] * 1e6).values, rng))
    return rows


def report(rows, heading):
    print(f"\n{heading}")
    print(f"{'config':<20}{'metric':<14}{'n':>5}{'median':>12}"
          f"{'SEM(mean)':>12}{'SE(median)':>12}{'ratio':>8}")
    print("-" * 83)
    for r in rows:
        print(f"{r['config']:<20}{r['metric']:<14}{r['n']:>5}{r['median']:>12.5f}"
              f"{r['sem_of_mean']:>12.5f}{r['se_of_median']:>12.5f}{r['ratio']:>7.1f}x")


def main(rsa_csv: str = RSA_CSV, pqc_csv: str = PQC_CSV):
    rng = np.random.default_rng(SEED)
    print(f"Bootstrap standard error of the median "
          f"({N_RESAMPLES} resamples, seed {SEED})")
    print("ratio = SEM(mean) / SE(median).  >1 = published error bars are "
          "conservative;  <1 = too narrow.")

    all_rows = []

    if os.path.exists(rsa_csv):
        rows = analyse_rsa(rng, rsa_csv)
        report(rows, f"RSA  (from {rsa_csv}, 16-byte message)")
        all_rows += rows
    else:
        print(f"\n[skipped] {rsa_csv} not found.")

    if os.path.exists(pqc_csv):
        rows = analyse_pqc(rng, pqc_csv)
        report(rows, f"Kyber / FrodoKEM  (from {pqc_csv})")
        all_rows += rows
    else:
        print(f"\n[skipped] {pqc_csv} not found.")

    if not all_rows:
        raise SystemExit("No benchmark CSVs found -- run the benchmarks first.")

    conservative = [r for r in all_rows if r["ratio"] >= 1.0]
    too_narrow = [r for r in all_rows if r["ratio"] < 1.0]

    print(f"\n{'=' * 83}")
    print(f"{len(conservative)} of {len(all_rows)} measurements have "
          f"SEM >= SE(median)  (conservative).")
    if too_narrow:
        print(f"{len(too_narrow)} do NOT. The manuscript must not claim the bound "
              f"is conservative everywhere:")
        for r in too_narrow:
            print(f"    {r['config']:<20}{r['metric']:<14}ratio {r['ratio']:.1f}x")
    print("\nQuote in the Discussion only the range you can see above.")
    return all_rows


if __name__ == "__main__":
    main()

Writing bootstrap_median_se.py


# Section 6: Execution & Export

In [11]:
import subprocess

def run_step(script, allow_exit_codes=(0,)):
    result = subprocess.run(["python", script], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode not in allow_exit_codes:
        print(result.stderr)
        raise RuntimeError(f"{script} failed (exit {result.returncode}) — stopping pipeline.")
    elif result.returncode != 0:
        print(f"[NOTE] {script} exited {result.returncode} (expected/non-fatal) — continuing.\n")

#run_step("run_pqc_benchmark.py")
run_step("generate_pqc_figure3.py")
run_step("run_qsafe_simulation.py")
#run_step("build_figure6_drawio.py", allow_exit_codes=(0, 1))

Saved to figures/Figure 3 - Kyber vs FrodoKEM Performance.png

Numbers for the Results text (medians, from the same CSVs):
  Complete key exchange: Kyber768 0.0917 ms | FrodoKEM-976-AES 3.0671 ms
  Speed ratio: 33.4x

  Ciphertext sizes (bytes) -- protocol constants, so one value each:
    Kyber512               768
    Kyber768               1088
    Kyber1024              1568
    FrodoKEM-640-AES       9752
    FrodoKEM-976-AES       15792
    FrodoKEM-1344-AES      21696

  FrodoKEM / Kyber ciphertext size ratio, by security level:
    Kyber512 vs FrodoKEM-640-AES: 12.7x
    Kyber768 vs FrodoKEM-976-AES: 14.5x
    Kyber1024 vs FrodoKEM-1344-AES: 13.8x

Using median of measured times: Kyber768=0.0917 ms, FrodoKEM-976-AES=3.0671 ms (ratio 33.4x)
Simulation inputs (medians, straight from pqc_complete_key_exchange_results.csv -- not hand-entered):
  Kyber768: 0.0917 ms per exchange
  FrodoKEM-976-AES: 3.0671 ms per exchange
  Ratio: 33.4x

Saved to figures/Figure 4 - Q-Safe Rising Thre

### Figure 6 note

`build_figure6_drawio.py` (above) generates the **starter** .drawio XML for the Figure 6 schematic. Do **not** re-run its `main()` casually — if the .drawio file has been hand-edited since, regenerating would overwrite the starter. The final Figure 6 PNG is exported from draw.io, not from this notebook.

### Before Editorial Manager upload

Rename outputs to plain names: `Figure 3.png`, `Figure 4.png`, `Figure 5.png`. Table 2 is pasted into the manuscript as an editable Word table from `Table 2 - Q-Safe Simulation Results.csv`.

In [12]:
from google.colab import files
import os # Added os import

from pqc_config import (
    FIGURE_DIR
)

for f in ["pqc_benchmark_results.csv", "pqc_complete_key_exchange_results.csv",
          "Table 2 - Q-Safe Simulation Results.csv",
          os.path.join(FIGURE_DIR, "Figure 3 - Kyber vs FrodoKEM Performance.png"), # Used os.path.join
          os.path.join(FIGURE_DIR, "Figure 4 - Q-Safe Rising Threat Scenario.png"), # Used os.path.join
          os.path.join(FIGURE_DIR, "Figure 5 - Q-Safe Threat Spikes Scenario.png"), # Used os.path.join
          os.path.join(FIGURE_DIR, "qsafe_constrained_edge.png"), # Used os.path.join and added comma
          "benchmark_pqc.py", "plot_pqc_comparison.py", "qsafe_simulation.py", "bootstrap_median_se.py",
          "verify_pqc_results.py", "build_figure6_drawio.py"]:
    if os.path.exists(f):
        files.download(f)
    else:
        print(f"[SKIP] {f} not found")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>